# RAG(Retrieval Augmented Generation)
- [RAG](https://python.langchain.com/v0.1/docs/modules/data_connection/)은 *Retrieval Augmented Generation*의 약자로, **검색 기반 생성 기법**을 의미한다. 이 기법은 LLM이 특정 문서에 기반하여 보다 정확하고 신뢰할 수 있는 답변을 생성할 수 있도록 돕는다.     
- 사용자의 질문에 대해 자체적으로 구축한 데이터베이스(DB)나 외부 데이터베이스에서 질문과 관련된 문서를 검색하고, 이를 질문과 함께 LLM에 전달한다.
- LLM은 같이 전달된 문서를 바탕으로 질문에 대한 답변을 생성한다. 
- 이를 통해 LLM이 학습하지 않은 내용도 다룰 수 있으며, 잘못된 정보를 생성하는 환각 현상(*hallucination*)을 줄일 수 있다.

## RAG와 파인튜닝(Fine Tuning) 비교

### 파인튜닝(Fine Tuning)

- **정의**: 사전 학습(pre-trained)된 LLM에 특정 도메인의 데이터를 추가로 학습시켜 해당 도메인에 특화된 맞춤형 모델로 만드는 방식이다.
- **장점**
  - 특정 도메인에 최적화되어 높은 정확도와 성능을 낼 수 있다.
- **단점**
  - 모델 재학습에 많은 시간과 자원이 필요하다.
  - 새로운 정보가 반영되지 않으며, 이를 위해서는 다시 학습해야 한다.

### RAG

- **정의**: 모델을 다시 학습시키지 않고, 외부 지식 기반에서 정보를 검색하여 실시간으로 답변에 활용하는 방식이다.
- **장점**
  - 최신 정보를 쉽게 반영할 수 있다.
  - 모델을 수정하지 않아도 되므로 효율적이다.
- **단점**
  - 검색된 문서의 품질에 따라 답변의 정확성이 달라질 수 있다.
  - 검색 시스템 구축이 필요하다.

## 정리

| 항목       | 파인튜닝 | RAG |
| -------- | ---- | --- |
| 도메인 최적화  | 가능   | 제한적 |
| 최신 정보 반영 | 불가능  | 가능  |
| 구현 난이도   | 높음   | 보통  |
| 유연성      | 낮음   | 높음  |

- LLM은 학습 당시의 데이터만을 기반으로 작동하므로 최신 정보나 기업 내부 자료와 같은 특정한 지식 기반에 접근할 수 없다.
- 파인튜닝은 시간과 비용이 많이 들고 유지보수가 어렵다.
-	반면, RAG는 기존 LLM을 변경하지 않고도 외부 문서를 통해 그 한계를 보완할 수 있다.
- RAG는 특히 빠르게 변화하는 정보를 다루는 분야(예: 기술 지원, 뉴스, 법률 등)에서 유용하게 활용된다. 반면, 정적인 정보에 대해 높은 정확도가 필요한 경우에는 파인튜닝이 효과적이다.


## RAG 작동 단계
- 크게 "**정보 저장(인덱싱)**", "**검색**, **생성**"의 단계로 나눌 수 있다.
  
### 1. 정보 저장(인덱싱)
RAG는 사전에 정보를 가공하여 **벡터 데이터베이스**(Vector 저장소)에 저장해 두고, 나중에 검색할 수 있도록 준비한다. 이 단계는 다음과 같은 과정으로 이루어진다.

1. **Load (불러오기)**
   - 답변시 참조할 사전 정보를 가진 데이터들을 불러온다.
2. **Split/Chunking (문서 분할)**
   - 긴 텍스트를 일정한 길이의 작은 덩어리(*chunk*)로 나눈다.
   - 이렇게 해야 검색과 생성의 정확도를 높일 수 있다.
3. **Embedding (임베딩)**
   - 각 텍스트 조각을 **임베딩 벡터**로 변환한다.
   - 임베딩 벡터는 그 문서의 의미를 벡터화 한 것으로 질문과 유사한 문서를 찾을 때 인덱스로 사용된다.
4. **Store (저장)**
   - 임베딩된 벡터를 **벡터 데이터베이스**(벡터 저장소)에 저장한다.
   - 벡터 데이터베이스는 유사한 질문이나 문장을 빠르게 찾을 수 있도록 특화된 데이터 저장소이다.
   
![rag](figures/rag1.png)

### 2. 검색, 생성

사용자가 질문을 하면 다음과 같은 절차로 답변이 생성된다.
1. **Retrieve (검색)**
   - 사용자의 질문을 임베딩한 후, 이 질문 벡터와 유사한 context 벡터를 벡터 데이터베이스에서 검색하여 찾는다.
2. **Query (질의 생성)**
   - 벡터 데이터베이스에서 검색된 문서 조각과 사용자의 질문을 함께 **프롬프트**(prompt)로 구성하여 LLM에 전달한다.
3. **Generation (응답 생성)**
   - LLM은 받은 프롬프트에 대한 응답을 생성한다.
   
- **RAG 흐름**
  
![Retrieve and Generation](figures/rag2.png)


# Document Loader
- LLM에게 질의할 때 같이 제공할 Data들을 저장하기 위해 먼저 읽어들인다.(Load)
- 데이터 Resouce는 다양하다.
    - 데이터를 로드(load)하는 방식은 저장된 위치와 형식에 따라 다양하다. 
      - 로컬 컴퓨터(Local Computer)에 저장된 문서
        - 예: CSV, Excel, JSON, TXT 파일 등
      - 데이터베이스(Database)에 저장된 데이터셋
      - 인터넷에 존재하는 데이터
        - 예: 웹에 공개된 API, 웹 페이지에 있는 데이터, 클라우드 스토리지에 저장된 파일 등

![rag_load](figures/rag_load.png)

- 다양한 문서 형식(format)에 맞춰 읽어오는 다양한 **document loader** 들을 Langchain에서 지원한다.
    - 다양한 Resource들로 부터 데이터를 읽기 위해서는 다양한 라이브러리를 이용해 서로 다른 방법으로 읽어야 한다.
    - Langchain은 데이터를 읽는 다양한 방식의 코드를 하나의 interface로 사용 할 수 있도록 지원한다.
        - https://python.langchain.com/docs/how_to/#document-loaders
    - 다양한 3rd party library(ppt, github 등등 다양한 3rd party lib도 있음. )들과 연동해 다양한 Resource로 부터 데이터를 Loading 할 수 있다.
        - https://python.langchain.com/docs/integrations/document_loaders/
- **모든 document loader는 기본적으로 동일한 interface(사용법)로 호출할 수있다.**
- **반환타입**
    - **list[Document]**
    - Load 한 문서는 Document객체에 정보들을 넣는다. 여러 문서를 읽을 수 있기 대문에 list에 묶어서 반환한다.
        - **Document 속성**
            - page_content: 문서의 내용
            - metadata(option): 문서에 대한 메타데이터(정보)를 dict 형태로 저장한다. 
            - id(option): 문서의 고유 id
     
- **주의**
    - Langchain을 이용해 RAG를 구현할 때 **꼭 Langchain의 DocumentLoader를 사용해야 하는 것은 아니다.**
    - DocumentLoader는 데이터를 읽어오는 것을 도와주는 라이브러리일 뿐이다. 다른 라이브러리를 이용해서 읽어 들여도 상관없다. 

## 주요 Document Loader

### Text file
- TextLoader 이용

In [5]:
from langchain_community.document_loaders import TextLoader

path = "data/olympic.txt"

# 객체 생성 - 읽어들일 자원(파일)의 위치
loader = TextLoader(path, encoding='utf-8')

# load - 읽어오기
docs = loader.load() # 메소드 호출 시 읽음.
# docs = loader.lazy_load() : 읽은 문서를 사용(조회)할 때 읽음.

print(type(docs), len(docs))

<class 'list'> 1


In [6]:
print(docs[0].page_content[:100])

올림픽
올림픽(영어: Olympic Games, 프랑스어: Jeux olympiques)은 전 세계 각 대륙 각국에서 모인 수천 명의 선수가 참가해 여름과 겨울에 스포츠 경기를 하


In [7]:
print(docs[0].metadata)
docs[0].metadata['category']= "스포츠"
print(docs[0].metadata)

{'source': 'data/olympic.txt'}
{'source': 'data/olympic.txt', 'category': '스포츠'}


In [9]:
print(docs[0].metadata)
docs[0].metadata['category']= "스포츠"
docs[0].metadata["tag"]="올림픽",""
print(docs[0].metadata)

{'source': 'data/olympic.txt', 'category': '스포츠'}
{'source': 'data/olympic.txt', 'category': '스포츠', 'tag': ('올림픽', '')}


### PDF
- PyPDF, Pymupdf 등 다양한 PDF 문서를 읽어들이는 파이썬의  3rd party library들을 이용해 pdf 문서를 Load 한다.
    - https://python.langchain.com/docs/integrations/document_loaders/#pdfs
- 각 PDF Loader 특징
    -  PyMuPDFLoader
        -   텍스트 뿐 아니라 이미지, 주석등의 정보를 추출하는데 성능이 좋다.
        -   PyMuPDF 라이브러리 기반
    - PyPDFLoader
        - 텍스트를 빠르게 추출 할 수있다.
        - PyPDF2 라이브러리 기반. 경량 라이브러리로 빠르고 큰 파일도 효율적으로 처리한다.
    - PDFPlumberLoader
        - 표와 같은 복잡한 구조의 데이터 처리하는데 강력한 성능을 보여준다. 텍스트, 이미지, 표 등을 모두 추출할 수 있다. 
        - PDFPlumber 라이브러리 기반
- 설치 패키지
    - DocumentLoader와 연동하는 라이브러리들을 설치 해야 한다.
    - `pip install pypdf -qU`
    - `pip install pymupdf -qU`
    - `pip install pdfplumber -qU`

In [ ]:
# !uv pip install pypdf pymupdf pdfplumber

Resolved 10 packages in 324ms
Prepared 6 packages in 892ms
Installed 8 packages in 297ms
 + cffi==2.0.0
 + cryptography==46.0.3
 + pdfminer-six==20251107
 + pdfplumber==0.11.8
 + pycparser==2.23
 + pymupdf==1.26.7
 + pypdf==6.5.0
 + pypdfium2==5.2.0


In [17]:
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader, PDFPlumberLoader

path = "data/novel/동백꽃_김유정.pdf"
loader = PDFPlumberLoader(path)
# loader = PyMuPDFLoader(path)
# loader = PyPDFLoader(path, mode="single") #mode = single : 한 개의 문서로 읽음.(default : page, page별로 doc를 만든다)
docs = loader.load()

Could get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could get FontBB

In [18]:
docs[0].metadata

{'source': 'data/novel/동백꽃_김유정.pdf',
 'file_path': 'data/novel/동백꽃_김유정.pdf',
 'page': 0,
 'total_pages': 16,
 'Author': 'Unknown',
 'CreationDate': "D:20241124070355+00'00'",
 'Creator': 'Wikisource',
 'ModDate': "D:20241124070356+00'00'",
 'Producer': 'Wikisource',
 'Title': '동백꽃'}

In [19]:
print(len(docs))
# PyPDFLoader(mode = page) = page 단위로 잘라낸 것임. (원본 16페이지임)
for doc in docs :
    doc.metadata['author'] = '김유정'
docs[0].metadata

16


{'source': 'data/novel/동백꽃_김유정.pdf',
 'file_path': 'data/novel/동백꽃_김유정.pdf',
 'page': 0,
 'total_pages': 16,
 'Author': 'Unknown',
 'CreationDate': "D:20241124070355+00'00'",
 'Creator': 'Wikisource',
 'ModDate': "D:20241124070356+00'00'",
 'Producer': 'Wikisource',
 'Title': '동백꽃',
 'author': '김유정'}

In [13]:
docs[1].metadata

{'producer': 'Wikisource',
 'creator': 'Wikisource',
 'creationdate': '2024-11-24T07:03:55+00:00',
 'source': 'data/novel/동백꽃_김유정.pdf',
 'file_path': 'data/novel/동백꽃_김유정.pdf',
 'total_pages': 16,
 'format': 'PDF 1.4',
 'title': '동백꽃',
 'author': '김유정',
 'subject': '',
 'keywords': '',
 'moddate': '2024-11-24T07:03:56+00:00',
 'trapped': '',
 'modDate': "D:20241124070356+00'00'",
 'creationDate': "D:20241124070355+00'00'",
 'page': 1}

In [10]:
print(docs[0].page_content[:500])

1 
동백꽃
Exported from Wikisource on 2024 년  11 월  24 일
2 
오늘도  또  우리  수탉이  막  쫓기었다 . 내가  점심을  먹고  나무
를  하러  갈  양으로  나올  때이었다 . 산으로  올라서려니까  등
뒤에서  푸드득푸드득 , 하고  닭의  횃소리가  야단이다 . 깜짝
놀라서  고개를  돌려보니  아니나다르랴 , 두  놈이  또  얼리었
다 .
점순네  수탉 ( 은  대강이가  크고  똑  오소리같이  실팍하게  생
긴  놈 ) 이  덩저리  작은  우리  수탉을  함부로  해내는  것이다 . 그
것도  그냥  해내는  것이  아니라  푸드득하고  면두를  쪼고  물러
섰다가  좀  사이를  두고  푸드득하고  모가지를  쪼았다 . 이렇게
멋을  부려  가며  여지없이  닦아  놓는다 . 그러면  이  못생긴  것
은  쪼일  적마다  주둥이로  땅을  받으며  그  비명이  킥 , 킥 , 할
뿐이다 . 물론  미처  아물지도  않


### CSVLoader

In [20]:
from langchain_community.document_loaders import CSVLoader

path = "data/boston_hosing.csv"
loader = CSVLoader(path)
docs = loader.load() # 행 단위로 Document를 생성

print(len(docs))

506


In [23]:
docs[0].metadata
docs[10].metadata

{'source': 'data/boston_hosing.csv', 'row': 10}

In [24]:
print(docs[10].page_content)

CRIM: 0.22489
ZN: 12.5
INDUS: 7.87
CHAS: 0.0
NOX: 0.524
RM: 6.377
AGE: 94.3
DIS: 6.3467
RAD: 5.0
TAX: 311.0
PTRATIO: 15.2
B: 392.52
LSTAT: 20.45
MEDV: 15.0


### Web 문서 로드

#### WebBaseLoader를 이용해 Web 문서로딩

requests와 BeautifulSoup을 이용해 web 페이지의 내용을 크롤링해서 Document로 loading한다.

- 주요 파라미터
  - **web_paths***: str | list[str]
    - 크롤링할 대상 URL
  - **requests_kwargs**: dict
    - requests.get() 에 전달할 파라미터를 dict로 전달. (key: parameter변수명, value: 전달할 값)
    - headers, cookies, verify 등 설정 전달
  - **header_template**: dict
    - HTTP Header 에 넣을 값을 dict 로 전달.
  - **encoding**
    - requests의 응답 encoding을 설정 (bs_kwargs의 from_encoding 보다 상위에서 적용됨)
  - **bs_kwargs**
    - BeautifulSoup initializer에 전달할 파라미터를 dict로 전달. (key: parameter변수명, value: 전달할 값)
    -  주요 옵션
       - **parse_only**: 요청 페이지에서 특정 요소만 선택해서 가져오기. **SoupStrainer를 사용**한다.
         - BeautifulSoup의 `SoupStrainer` 를 이용해 페이지의 일부분만 가져오기
           - 웹 페이지를 파싱(parse, 구조 분석)할 때, 페이지 전체가 아닌 특정 부분만 필요한 경우가 많다. BeautifulSoup 라이브러리의 SoupStrainer를 사용하면, 원하는 태그나 속성이 있는 요소만 골라서 파싱할 수 있다.
           - BeautifulSoup("html문서", parse_only=Strainer객체)
               - Strainer객체에 지정된 영역에서만 내용 찾는다.
           - `SoupStrainer("태그명")`, `SoupStrainer(["태그명", "태그명"])`
             - 지정한 태그 만 조회
           - `SoupStrainer(name="태그명", attrs={속성명:속성값})`
             -  지정한 태그 중 속성명=속성값인 것만 조회
        - **from_encoding**: Encoding 설정 
          - "from_encoding":"utf-8"
   - **bs_get_text_kwargs**:
     - BeautifulSoup객체.get_text() 에 전달할 파라미터 dict로 전달. (key: parameter변수명, value: 전달할 값)
     - **RAG 구축시 `separator` 와 `strip=True` 으로 설정하는 것이 좋다.** (RAG 품질을 위해 강력히 권장되는 설정이다.)
       -  get_text() 는 기본적으로 태그를 제거하고 텍스트만 이어 붙여 반환한다. `separator=구분자문자` 를 지정하여 추출된 텍스트 요소들 사이에 원하는 구분자를 지정할 수있다. `\n` 을 구분자로 사용하면 텍스트 블록 사이에 줄바꿈이 들어가 **문단의 구조를 어느정도 살릴 수 있다.**
       -  웹 문서의 줄바꿈도 포함해서 읽기 때문에 공백과 줄바꿈이 혼재된 상태로 반환된다. `strip=True`로 설정하면 추출된 문자 앞뒤의 공백 문자들을 제거할 수있다.

In [ ]:
# !uv pip install requests beautifulsoup4 lxml

Resolved 9 packages in 197ms
Prepared 2 packages in 63ms
Installed 3 packages in 74ms
 + beautifulsoup4==4.14.3
 + lxml==6.0.2
 + soupsieve==2.8.1


In [29]:
from bs4 import BeautifulSoup

html_txt = """
<html>
<body>
<p>
<b>제목</b>
<span>내용</span>
</p>
<p>다음 문단</p>
<div>다음 내용</div>
</body>
</html>
"""

soup = BeautifulSoup(html_txt)
# 태그 빼고 text를 추출할 때 get_text()사용
txt1 = soup.get_text()
print(txt1)

txt2 = soup.get_text(strip=True) # 좌우 공백문자(공백, 엔터) 제거
print(txt2)

txt3 = soup.get_text(strip=True, separator="\n\n") # 각 태그의 text를 지정한 구분자로 나눈다.

print(txt3)




제목
내용

다음 문단
다음 내용



제목내용다음 문단다음 내용
제목

내용

다음 문단

다음 내용


In [31]:
import os
# chrome : my user agent 검색
# USER AGENT를 환경변수에 등록
os.environ["USER_AGENT"] = "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/142.0.0.0 Safari/537.36"

In [33]:
from langchain_community.document_loaders import WebBaseLoader

urls = ["https://m.entertain.naver.com/home/article/468/0001204002",
        "https://m.entertain.naver.com/home/article/311/0001955231"]


loader = WebBaseLoader(
    web_path=urls,
    default_parser="lxml" # BeautifulSoup(문서, 'lxml')
)

docs = loader.load()
print(len(docs))

2


In [34]:
from pprint import pprint
pprint(docs[0].metadata)

{'language': 'ko',
 'source': 'https://m.entertain.naver.com/home/article/468/0001204002',
 'title': '김우빈♥신민아, 결혼식 사진 최초 공개 ‘눈부시게 아름다워’ [공식]'}


In [35]:
print(docs[0].page_content)

김우빈♥신민아, 결혼식 사진 최초 공개 ‘눈부시게 아름다워’ [공식]본문 바로가기NAVER엔터뉴스스포츠네이버톡홈드라마영화뮤직연애포토랭킹최신뉴스연재김우빈♥신민아, 결혼식 사진 최초 공개 ‘눈부시게 아름다워’ [공식]입력2025.12.22. 오전 9:19기사원문공감좋아요0응원해요0축하해요0기대해요0놀랐어요0슬퍼요0텍스트 음성 변환 서비스본문 듣기를 종료하였습니다.글자 크기 변경공유하기사진 | 에이엠엔터테인먼트[스포츠서울 | 박경호 기자] 배우 신민아와 김우빈의 결혼식 사진이 공개됐다.22일 소속사 에이엠엔터테인먼트는 지난 20일 오후 7시 서울 중구 신라호텔 다이너스티홀에서 열린 신민아와 김우빈의 웨딩 본식 사진을 공개했다.공개된 사진 속 신민아는 마치 눈꽃을 연상시키는 장식의 튜브톱 드레스를 입고 환한 미소를 짓고 있다. 김우빈 역시 클래식한 블랙 턱시도를 완벽한 피지컬로 소화한 모습이다.사진 | 에이엠엔터테인먼트이날 예식은 평소 김우빈과 절친한 배우 이광수의 사회로 시작해 주례를 맡은 법륜스님의 따뜻하고도 깊이 있는 덕담이 현장을 훈훈하게 물들였다. 또 축가로는 가수 카더가든이 등장해 신민아가 출연한 드라마 ‘갯마을 차차차’의 삽입곡 ‘로맨틱 선데이’를 열창했다.오랜 시간 굳건한 사랑과 신뢰를 이어온 신민아와 김우빈은 앞으로 배우로서 활발한 작품 활동을 이어가는 것은 물론, 사회에 선한 영향력을 전하는 부부로서 대중을 만날 예정이다. park5544@sportsseoul.com본문의 검색 링크는 AI 자동 인식으로 제공됩니다. 일부에 대해서는 미제공될 수 있고 동일한 명칭이 다수 존재하는 경우에는 전체 검색 결과로 연결될 수 있습니다. 오분류 제보하기박경호 기자구독 0응원 0구독김남일, 결혼 18년 만에 일냈네...강남 한복판서 햄버거 사업폰세 아내 “첫 딸, 말 떼면 ‘한화’ 외칠 것 같아” (올해의상) [SS영상]스포츠서울언론사홈 바로가기Copyright ⓒ 스포츠서울. All rights reserved. 무단 전재 및 재배포 금지.이 기사는 언론사에서 

In [36]:
from bs4 import SoupStrainer
# SoupStrainer
# (name = "a") # a 태그들
# (name="a", attr={"href":"......"}) # 태그 + 속성 조건
# (id = "tag의 id"), # id로 조회.

loader2 = WebBaseLoader(
    web_path=urls,
    bs_kwargs={
        "parse_only":SoupStrainer(attrs={"class":["_article_content"]})
    },
    bs_get_text_kwargs={
        "separator" : "\n", "strip" : True
    }
)
docs2 = loader2.load()
len(docs2)

2

#### RecursiveUrlLoader

- 주어진 URL에서 시작하여 그 페이지 안의 내부 링크를 재귀적으로 따라가며 여러 웹 문서를 자동 수집하여 로드한다.
  - 시작 url을 요청/페이지를 파싱 한 뒤에 `<a href>` 들을 수집하고 그 페이지들을 요청/페이지 파싱을 한다. 
- WebBaseLoader가 단일 페이지(단일 URL) 단위라면 RecursiveUrlLoader는 **웹 사이트 구조 전체를 크롤링하는 전용 수집기**에 가깝다.
```bash
시작 URL
 ├─ 내부 링크 1
 │   ├─ 내부 링크 1-1
 │   └─ 내부 링크 1-2
 ├─ 내부 링크 2
 └─ 내부 링크 3
```
위 구조일때 무든 페이지를 재귀적으로 수집한다.
- 주요 파라미터
  - **url**: 시작 url
  - **max_depth**
    - 링크를 몇 단계 **깊이** 까지 따라갈지 제한
    - 사이트 폭주를 막기 위한 안전장치
      - **0**: 시작페이지만, **1**: 시작페이지 + 1차링크, **2**(기본값): 시작페이지 + 1차링크 + 2차링크
  - **exclude_dirs**: list[str]
    - 크롤링 제외 경로
    - ex) `exclude_dirs=['/login', 'signup']`
  - **prevent_outside**: bool
    - True: base_url 바깥 링크는 가져오지 않고 무시한다.
  - **base_url**: str
    - prevent_outside=True일 때 바깥링크의 기준. 없으면 `url`(시작 url)의 host가 된다. 
  - **extractor**
    - 문서 내용 추출 사용자 정의 함수
    - default는 응답 받은 페이지를 `BeautifulSoup(응답페이지).get_text()` 로 텍스트를 추출한다.
    - ````python
        def custom_extractor(html:str) ->str:
            # 웹 페이지 문서를 입력으로 받는다.
            soup = BeautifulSoup(html, 'lxml')
            return soup.select_one('article').get_text() # 원하는 항목을 추출해서 반환한다.
        
        loader = RecursiveUrlLoader(
            url=start_url,
            extractor=custom_extractor
        )    
```

In [ ]:
from bs4 import BeautifulSoup
from langchain_community.document_loaders import RecursiveUrlLoader

def extractor(html:str)->str:
    # 전체 페이지를 받아서 원하는 부분만 parsing 한 뒤에 반환.
    soup = BeautifulSoup(html, "lxml")
    body = soup.select_one("div.body")
    return body.get_text(strip=True, separator="\n") if body else soup.get_text(strip=True, separator="\n")


url = "https://docs.python.org/3"
loader = RecursiveUrlLoader(
    url=url,
    extractor=extractor,
    max_depth=2, # default : 2
    prevent_outside=True, # url 외부 링크는 가져오지 않도록 한다.(default가 url의 host.)
    base_url=url
)


In [45]:
docs = loader.load()

In [46]:
print(len(docs))

18


In [48]:
idx=10
pprint(docs[idx].metadata)

{'content_type': 'text/html',
 'language': None,
 'source': 'https://docs.python.org/3.2/',
 'title': 'Overview — Python v3.2.6 documentation'}


In [49]:
print(docs[idx].page_content)

Python v3.2.6 documentation
Welcome! This is
    the documentation for Python
    3.2.6, last updated Oct 12, 2014.
Parts of the documentation:
What's new in Python 3.2?
or
all "What's new" documents
since 2.0
Tutorial
start here
Library Reference
keep this under your pillow
Language Reference
describes syntax and language elements
Python Setup and Usage
how to use Python on different platforms
Python HOWTOs
in-depth documents on specific topics
Extending and Embedding
tutorial for C/C++ programmers
Python/C API
reference for C/C++ programmers
Installing Python Modules
information for installers & sys-admins
Distributing Python Modules
sharing modules with others
FAQs
frequently asked questions (with answers!)
Indices and tables:
Global Module Index
quick access to all modules
General Index
all functions, classes, terms
Glossary
the most important terms explained
Search page
search this documentation
Complete Table of Contents
lists all sections and subsections
Meta information:
Report

### ArxivLoader
- https://github.com/lukasschwab/arxiv.py
- [arXiv-아카이브](https://arxiv.org/) 는 미국 코렐대학에서 운영하는 **무료 논문 저장소**로, 물리학, 수학, 컴퓨터 과학, 생물학, 금융, 경제 등 **과학, 금융 분야의 논문**들을 공유한다.
- `ArxivLoader` 를 사용해 원하는 주제의 논문들을 arXiv에서 가져와 load할 수 있다.
- **arXiv API**를 사용해 논문을 가져올 수 있다.
  - https://python.langchain.com/api_reference/community/document_loaders/langchain_community.document_loaders.arxiv.ArxivLoader.html
- 설치
  - `pip install langchain-community -qU`
  - `pip install arxiv -qU`



In [ ]:
# !uv pip install arxiv
# !uv pip install pip-system-certs

Resolved 2 packages in 110ms
Prepared 1 package in 25ms
Installed 2 packages in 140ms
 + pip==25.3
 + pip-system-certs==5.3


In [1]:
import pip_system_certs

In [2]:
# arxiv lib 사용
import arxiv
search = arxiv.Search(
    query="Advanced RAG", # 검색어
    max_results=5, # 검색 논문 최대 개수
    sort_by=arxiv.SortCriterion.LastUpdatedDate, # 정렬 기준
)

# LastUpdatedDate : 논문이 마지막으로 수정된 날짜 기준
# Relevance : qeury와 관련성이 높은 순서.
# SubmittedDate : 논문이 처음 제출된 날짜 기준

# 검색처리 Client
client = arxiv.Client()
results = client.results(search) # 검색 (iterator)

In [3]:
print(type(results)) # iterator : next() or for in 으로 가져올 수 있음.

#  첫번째 것만 조회
paper = next(results)


<class 'itertools.islice'>


In [7]:
# 논문 정보
print(paper.title) # 제목
print(paper.authors) # 논문 저자
print(paper.authors[0].name) # Authors.name : 이름 추출
print(paper.summary) # 논문 요약(초록)
print(paper.pdf_url) # arxiv의 논문 url
print(paper.get_short_id()) # arivx 내 이 논문의 id

Keypoint Counting Classifiers: Turning Vision Transformers into Self-Explainable Models Without Training
[arxiv.Result.Author('Kristoffer Wickstrøm'), arxiv.Result.Author('Teresa Dorszewski'), arxiv.Result.Author('Siyan Chen'), arxiv.Result.Author('Michael Kampffmeyer'), arxiv.Result.Author('Elisabeth Wetzer'), arxiv.Result.Author('Robert Jenssen')]
Kristoffer Wickstrøm
Current approaches for designing self-explainable models (SEMs) require complicated training procedures and specific architectures which makes them impractical. With the advance of general purpose foundation models based on Vision Transformers (ViTs), this impracticability becomes even more problematic. Therefore, new methods are necessary to provide transparency and reliability to ViT-based foundation models. In this work, we present a new method for turning any well-trained ViT-based model into a SEM without retraining, which we call Keypoint Counting Classifiers (KCCs). Recent works have shown that ViTs can automatic

In [8]:
# 논문 저장
import os
os.makedirs('data/papers', exist_ok=True)

paper.download_pdf(dirpath='data/papers', filename=f"{paper.get_short_id()}.pdf")

'data/papers\\2512.17891v1.pdf'

In [9]:
# 전체 다운로드
for paper in results :
    paper.download_pdf(dirpath="data/papers", filename=f"{paper.get_short_id()}.pdf")

In [10]:
# langchain - ArxivLoader
from langchain_community.document_loaders import ArxivLoader

loader = ArxivLoader(
    query="RAG",
    top_k_results = 10,
)

docs = loader.load()

In [11]:
print(len(docs))

10


In [12]:
docs[0].metadata

{'Published': '2025-05-31',
 'Title': 'RAG-Gym: Systematic Optimization of Language Agents for Retrieval-Augmented Generation',
 'Authors': 'Guangzhi Xiong, Qiao Jin, Xiao Wang, Yin Fang, Haolin Liu, Yifan Yang, Fangyuan Chen, Zhixing Song, Dengyu Wang, Minjia Zhang, Zhiyong Lu, Aidong Zhang',
 'Summary': 'Retrieval-augmented generation (RAG) has shown great promise for knowledge-intensive tasks and recently advanced with agentic RAG, where language agents engage in multi-round interactions with external knowledge sources for adaptive information retrieval. However, existing agentic RAG methods often depend on ad-hoc prompt engineering and lack a unified optimization framework. We introduce RAG-Gym, a comprehensive platform that systematically explores three optimization dimensions: (1) prompt engineering, (2) actor tuning, and (3) critic training. For prompt engineering, we propose Re$^2$Search, a novel agent incorporating reasoning reflection that significantly outperforms standard p

In [ ]:
print(docs[0].page_content) # 논문 내용
# 다운로드 기능은 없음. 

arXiv:2502.13957v2  [cs.CL]  31 May 2025
RAG-Gym: Systematic Optimization of Language
Agents for Retrieval-Augmented Generation
Guangzhi Xiong∗1, Qiao Jin∗2, Xiao Wang3, Yin Fang2, Haolin Liu1, Yifan Yang2, Fangyuan
Chen4, Zhixing Song5, Dengyu Wang6, Minjia Zhang3, Zhiyong Lu†2, and Aidong Zhang†1
1University of Virginia, 2National Institutes of Health, 3University of Illinois at Urbana Champaign,
4Dana-Farber Cancer Institute, 5University of Alabama at Birmingham, 6Yale School of Medicine
Abstract
Retrieval-augmented generation (RAG) has shown great promise for knowledge-
intensive tasks and recently advanced with agentic RAG, where language agents
engage in multi-round interactions with external knowledge sources for adaptive
information retrieval. However, existing agentic RAG methods often depend on
ad-hoc prompt engineering and lack a unified optimization framework. We in-
troduce RAG-Gym, a comprehensive platform that systematically explores three
optimization dimensions: (1) pr

### Docling
- IBM Research에서 개발한 오픈소스 문서처리 도구로 다양한 종류의 문서를 구조화된 데이터로 변환해 생성형 AI에서 활용할 수있도록 지원한다.
- **주요기능**
  - PDF, DOCX, PPTX, XLSX, HTML, 이미지 등 여러 형식을 지원
  - PDF의 **페이지 레이아웃, 읽기 순서, 표 구조, 코드, 수식** 등을 분석하여 정확하게 읽어들인다.
  - OCR을 지원하여 스캔된 PDF나 이미지에서 텍스트를 추출할 수있다.
  - 읽어들인 내용을 markdown, html, json등 다양한 형식으로 출력해준다.
- 설치 : `pip install langchain-docling ipywidgets -qU` 
- 참조
  - docling 사이트: https://github.com/docling-project/docling
  - 랭체인-docling https://python.langchain.com/docs/integrations/document_loaders/docling/

In [ ]:
# !uv pip install langchain-docling transformers ipywidgets

# 딥러닝 모델 사용. GPU가 있을 경우 torch cuda버전을 먼저 설치 

Resolved 126 packages in 3.98s
   Building antlr4-python3-runtime==4.9.3
   Building pylatexenc==2.10
      Built pylatexenc==2.10
      Built antlr4-python3-runtime==4.9.3
Prepared 39 packages in 10.39s
Uninstalled 2 packages in 67ms
Installed 48 packages in 750ms
 + accelerate==1.12.0
 + antlr4-python3-runtime==4.9.3
 + colorlog==6.10.1
 + dill==0.4.0
 + docling==2.65.0
 + docling-core==2.57.0
 + docling-ibm-models==3.10.3
 + docling-parse==4.7.2
 + et-xmlfile==2.0.0
 + faker==39.0.0
 + jsonlines==4.0.0
 + jsonref==1.1.0
 + langchain-docling==2.0.0
 + latex2mathml==3.78.1
 + markdown-it-py==4.0.0
 + marko==2.2.1
 + mdurl==0.1.2
 + mpire==2.10.2
 + multiprocess==0.70.18
 + omegaconf==2.3.0
 + opencv-python==4.11.0.86
 + openpyxl==3.1.5
 - pillow==12.0.0
 + pillow==11.3.0
 + pluggy==1.6.0
 + polyfactory==3.2.0
 + pyclipper==1.4.0
 + pylatexenc==2.10
 - pypdfium2==5.2.0
 + pypdfium2==4.30.0
 + python-docx==1.2.0
 + python-pptx==1.0.2
 + pywin32==311
 + rapidocr==3.4.5
 + rich==14.2.0
 +

In [2]:
import os
# hugging face 로그인 - 모델 받기 위해서
from dotenv import load_dotenv
from huggingface_hub import login

hf_key = os.getenv("HUGGINGFACE_API_KEY")
login(hf_key)

In [ ]:
# !uv pip install accelerate

Audited 1 package in 11ms


In [3]:
from langchain_docling import DoclingLoader
from langchain_docling.loader import ExportType
path = "data/papers/2507.01939v4.pdf"
loader = DoclingLoader(
    file_path=path,
    export_type=ExportType.MARKDOWN
)

docs = loader.load()

2025-12-22 14:38:33,188 - INFO - detected formats: [<InputFormat.PDF: 'pdf'>]
2025-12-22 14:38:33,243 - INFO - Going to convert document batch...
2025-12-22 14:38:33,244 - INFO - Initializing pipeline for StandardPdfPipeline with options hash e15bc6f248154cc62f8db15ef18a8ab7
2025-12-22 14:38:33,255 - INFO - Loading plugin 'docling_defaults'
2025-12-22 14:38:33,255 - WARNING - The plugin langchain_docling will not be loaded because Docling is being executed with allow_external_plugins=false.
2025-12-22 14:38:33,255 - INFO - Registered picture descriptions: ['vlm', 'api']
2025-12-22 14:38:33,274 - INFO - Loading plugin 'docling_defaults'
2025-12-22 14:38:33,282 - WARNING - The plugin langchain_docling will not be loaded because Docling is being executed with allow_external_plugins=false.
2025-12-22 14:38:33,282 - INFO - Registered ocr engines: ['auto', 'easyocr', 'ocrmac', 'rapidocr', 'tesserocr', 'tesseract']
2025-12-22 14:38:33,289 - INFO - rapidocr cannot be used because onnxruntime i

In [4]:
len(docs)

1

In [5]:
print(docs[0].metadata)

{'source': 'data/papers/2507.01939v4.pdf'}


In [7]:
print(docs[0].page_content)

## SpecCLIP: Aligning and Translating Spectroscopic Measurements for Stars

XIAOSHENG ZHAO , 1, 2, 3 , ∗ YANG HUANG , 1, 2 GUIRONG XUE , 4, ∗ XIAO KONG , 2, 1 , ∗ JIFENG LIU , 2, 1 XIAOYU TANG, 5 TIMOTHY C. BEERS , 6, 7 YUAN-SEN TING , 8, 9 AND A-LI LUO 2, 1

1 School of Astronomy and Space Science, University of Chinese Academy of Sciences, Beijing 100049, People's Republic of China

2 National Astronomical Observatories, Chinese Academy of Sciences, Beijing 100012, People's Republic of China

3 Department of Physics &amp; Astronomy, The Johns Hopkins University, Baltimore, MD 21218, USA

4 Zhejiang Laboratory, Hangzhou 311121, People's Republic of China

5 Research Center for Astronomical Computing, Zhejiang Laboratory, Hangzhou 311121, People's Republic of China

6 Department of Physics and Astronomy, University of Notre Dame, Notre Dame, IN 46556, USA

7 Joint Institute for Nuclear Astrophysics - Center for the Evolution of the Elements (JINA-CEE), USA

8 Department of Astronomy, T

In [6]:
from IPython.display import Markdown

Markdown(docs[0].page_content)

## SpecCLIP: Aligning and Translating Spectroscopic Measurements for Stars

XIAOSHENG ZHAO , 1, 2, 3 , ∗ YANG HUANG , 1, 2 GUIRONG XUE , 4, ∗ XIAO KONG , 2, 1 , ∗ JIFENG LIU , 2, 1 XIAOYU TANG, 5 TIMOTHY C. BEERS , 6, 7 YUAN-SEN TING , 8, 9 AND A-LI LUO 2, 1

1 School of Astronomy and Space Science, University of Chinese Academy of Sciences, Beijing 100049, People's Republic of China

2 National Astronomical Observatories, Chinese Academy of Sciences, Beijing 100012, People's Republic of China

3 Department of Physics &amp; Astronomy, The Johns Hopkins University, Baltimore, MD 21218, USA

4 Zhejiang Laboratory, Hangzhou 311121, People's Republic of China

5 Research Center for Astronomical Computing, Zhejiang Laboratory, Hangzhou 311121, People's Republic of China

6 Department of Physics and Astronomy, University of Notre Dame, Notre Dame, IN 46556, USA

7 Joint Institute for Nuclear Astrophysics - Center for the Evolution of the Elements (JINA-CEE), USA

8 Department of Astronomy, The Ohio State University, 140 West 18th Avenue, Columbus, OH 43210, USA

9 Center for Cosmology and AstroParticle Physics (CCAPP), The Ohio State University, Columbus, OH 43210, USA

## ABSTRACT

In recent years, large language models (LLMs) have transformed natural language understanding through vast datasets and large-scale parameterization. Inspired by this success, we present SpecCLIP, a foundation model framework that extends LLM-inspired methodologies to stellar spectral analysis. Stellar spectra, akin to structured language, encode rich physical and chemical information about stars. By training foundation models on large-scale spectral datasets, our goal is to learn robust and informative embeddings that support diverse downstream applications. As a proof of concept, SpecCLIP involves pre-training on two spectral types-LAMOST low-resolution and Gaia XP-followed by contrastive alignment using the CLIP (Contrastive Language-Image Pre-training) framework, adapted to associate spectra from different instruments. This alignment is complemented by auxiliary decoders that preserve spectrum-specific information and enable translation (prediction) between spectral types, the former being achieved by maximizing mutual information between embeddings and input spectra. The result is a cross-spectrum framework that enables intrinsic calibration and flexible applications across instruments. We demonstrate that fine-tuning these models on moderate-sized labeled datasets improves adaptability to tasks such as stellar-parameter estimation and chemical-abundance determination. SpecCLIP also enhances the accuracy and precision of parameter estimates bench-marked against external survey data. In addition, its similarity search and cross-spectrum prediction capabilities offer potential for anomaly detection. Our results suggest that contrastively trained foundation models enriched with spectrum-aware decoders can advance precision stellar spectroscopy. Our code SpecCLIP is publicly available on GitHub /github .

Keywords: Galaxy: stellar content - stars: fundamental parameters - stars: distances - methods: data analysis

## 1. INTRODUCTION

Over the past decades, large-scale spectroscopic surveys have revolutionized our understanding of the formation and evolution of the Milky Way (Gilmore et al. 1989; Freeman &amp; Bland-Hawthorn 2002; Gray et al. 2002; Wyse 2009; Helmi 2020). These advances have been driven by three key forces. First, the continuous development of large-scale spec-

Corresponding author: Yang Huang huangyang@ucas.ac.cn

∗ These authors contributed equally to this work.

troscopic surveys - such as RAVE (Steinmetz et al. 2006), SEGUE (de Jong et al. 2010), APOGEE (Majewski et al. 2017), GALAH (De Silva et al. 2015), LAMOST (Zhao et al. 2012), and DESI (DESI Collaboration et al. 2016) - has provided an unprecedented volume of stellar spectra across diverse Galactic populations. Second, the creation of powerful data infrastructures (Helou et al. 1991; Szalay &amp; Gray 2001; Gray et al. 2002; Fitzpatrick et al. 2014; Moitinho et al. 2017), exemplified by the SkyServer (Szalay et al. 2001) and CasJobs (OMullane et al. 2005) systems built for the Sloan Digital Sky Survey (SDSS, Margon 1999; Abazajian et al. 2003, has democratized access to these datasets and enabled

efficient large-scale analyses. Third, the refinement of algorithms for extracting physical parameters from these spectra has enabled increasingly precise stellar characterization.

The latter effort encompasses traditional line-index methods, such as the SEGUE Stellar Parameter Pipeline (SSPP) (Lee et al. 2008); template-matching techniques, including UlySS (Koleva et al. 2009), the LAMOST stellar parameter pipeline (LASP; Wu et al. 2014) and the LAMOST stellar parameter pipeline at Peking University (LSP3; Xiang et al. 2015); as well as a range of machine learning approaches, among which the SSPP also incorporates a neural network module, alongside methods such as the Cannon (Ness et al. 2015), the Payne (Ting et al. 2017, 2019),the DD-Payne (Xiang et al. 2019), and the TransformerPayne (R´ o˙ za´ nski et al. 2025). All of these approaches, though diverse in methodology, rely heavily on supervision - either empirical or theoretical.

Empirical approaches are limited by the coverage of the reference libraries they use, even when these libraries are derived from fundamental measurements. For instance, the official LAMOST stellar parameter pipeline (LASP; Wu et al. 2014), which is based on the UlySS algorithm, can only measure iron abundances down to [Fe/H] = -2 . 5 , due to the limited parameter coverage of the ELODIE library (Moultaka et al. 2004). Theoretical approaches, while offering broader parameter coverage, are still subject to discrepancies between synthetic and observed spectra.

In addition, most existing pipelines are designed primarily to estimate atmospheric parameters, such as effective temperature ( T eff ), surface gravity (log g ) and iron abundance ([Fe/H]), in conjunction with a small set of elemental abundances. In contrast, determining other stellar properties, including reddening, stellar mass, and age, typically requires dedicated, task-specific pipelines. These efforts are further complicated by the diversity of spectral data in terms of wavelength coverage, resolution, and signal-to-noise ratio (SNR) - we refer to such quantities here as ' modalities '. Achieving consistency and placing all inferred parameters onto a uniform scale remains a significant challenge in the prevailing framework, where each parameter is commonly derived using a distinct, often non-overlapping model.

In parallel, the past five years have witnessed the remarkable success of large language models (LLMs) in natural language understanding, conversational AI, and text generation (Vaswani et al. 2017; Devlin et al. 2018; Radford et al. 2018, 2019; Brown et al. 2020). Breakthroughs in high-impact scientific domains - such as AlphaFold for protein structure prediction (Jumper et al. 2021) - have been enabled by the combination of massive datasets, large-scale models, and modern computational infrastructure (Kaplan et al. 2020).

Stellar spectra can be analogized to a structured language: their rich absorption features and overall shapes encode key information about a star's physical properties and evolutionary history. With the accumulation of millions of stellar spectra, it has become feasible to train foundation models (Leung &amp; Bovy 2024; Buck &amp; Schwarz 2024; Parker et al. 2024; Rizhko &amp; Bloom 2024; Smith et al. 2024; Zhong et al. 2024; Euclid Collaboration et al. 2025; Pattnaik et al. 2025) on these data using techniques inspired by LLMs. Here, 'foundation models' refer to models pre-trained on large and diverse datasets. The broader the spectral distribution and parameter ranges used during pre-training, the better the model can learn the underlying structure of the 'spectral language'. Once pre-trained, these models can be quickly fine-tuned with a small set of high-quality labels to perform a variety of downstream tasks, such as parameter estimation.

A particularly promising framework for learning across modalities is the Contrastive Language-Image Pre-training (CLIP) algorithm (Radford et al. 2021), which aligns text and image representations via contrastive learning. CLIP jointly trains two encoders (one for each modality) by maximizing the similarity between representations of matched pairs and minimizing it for mismatched ones. The result is a shared embedding space that enables direct comparison between different modalities, allowing the model to retrieve or match one modality given the other, even for new inputs not seen during training.

When applied to stellar spectra (Buck &amp; Schwarz 2024; Parker et al. 2024; Rizhko &amp; Bloom 2024), CLIP-style models can align spectra from different instruments or modalities with other astrophysical measurements, allowing for more flexible downstream tasks such as parameter estimation and anomaly detection. However, a known limitation of CLIP is that it prioritizes the shared information between modalities, potentially removing modality-specific features that are still important for some downstream tasks (Shwartz Ziv &amp; LeCun 2024).

Partly motivated by this issue, we introduce the SpecCLIP project - a unified framework for cross-modal representation learning of stellar spectra. SpecCLIP begins with pre-training on two types of spectra: LAMOST (Cui et al. 2012) low-resolution spectra (LRS; Zhao et al. 2012) and Gaia XP spectra (De Angeli et al. 2023; Gaia Collaboration et al. 2023). These are aligned in a shared embedding space using a CLIP-like contrastive objective, enhanced by auxiliary decoders to preserve the mutual information between the learned embeddings and the input spectra.

Mutual information (MI, Barber &amp; Agakov 2003; Poole et al. 2019; Devon Hjelm et al. 2018; Sui et al. 2023; Ting

2025) 1 is a key concept in information theory that quantifies how much one variable tells us about another. It serves as a natural objective for representation learning. Although we do not explicitly compute MI between the spectra and their embeddings, we adopt a simple and effective strategy (Wang et al. 2022): increasing contrastive training with input reconstruction as a regularization mechanism, similar to using a decoder in an autoencoder to encourage informative representations.

Another interesting feature enabled by our model is spectrum-to-spectrum translation . If the embeddings capture physically meaningful and shared information between spectra of different modalities, then it should be possible to predict one modality from the other, given suitable (spectrato-spectra) supervision. In parallel, Buck &amp; Schwarz (2024) demonstrated a similar idea using contrastive learning between Gaia XP and RVS spectra, employing CNN and multilayer perceptron (MLP) architectures. In our framework, we combine contrastive training and cross-modal prediction, augmented by spectral reconstruction within a unified architecture, enabling both spectrum-to-spectrum and spectrumto-parameter applications.

The key contributions of this work are:

- We design distinct tokenization and model structures tailored to LAMOST LRS and Gaia XP spectra to improve representation learning.
- Wedemonstrate that our unified model enables both inmodal and cross-modal search, as well as cross-modal prediction.

The remainder of this paper is organized as follows. In Section 2, we introduce the SpecCLIP model, including the separate pre-trained models and the CLIP-based alignment. Section 3 describes the downstream tasks, including modeling and sample selection for parameter estimation. The results, including parameter inference, spectral retrieval, and cross-modal prediction, are presented in Section 5. Section 6 offers further discussion and Section 7 summarizes the work. Additional material is provided in the appendices: Appendix A (additional results); Appendix B (continuum fitting); Appendix C (normalizing flows for parameter inference); Appendix D (pre-training details); Appendix E (projection models and decoders); and Appendix G (summary of key hyper-parameters and configurations)

1 Formally, the mutual information between two continuous random variables X and Y is defined as

<!-- formula-not-decoded -->

where p ( x, y ) is the joint density, and p ( x ) , p ( y ) are the marginals. For discrete variables, the integral becomes a sum.

## 2. SPECCLIP

SpecCLIP, developed in this work, is a foundation model designed to align stellar spectra across different modalities, such as those with varying wavelength coverage, resolution, and SNRs, using a CLIP-inspired architecture. 2 Our approach builds upon transformer-based foundation models or MLP-based autoencoders tailored for specific spectral types. These are combined with contrastive learning to align different spectra using a standard contrastive loss, supplemented by additional modules to capture both shared and modalityspecific information. Once trained, SpecCLIP enables various downstream tasks with either branch (not combination) of modalities, using a relatively small number of labeled examples (i.e., few-shot learning, as referred to in the literature).

Figure 1 provides an overview of the SpecCLIP framework. Foundation models are first independently pre-trained on each spectral modality in an unsupervised fashion. These models are then aligned using contrastive learning, with additional decoders added to enhance information retention. After this process, the final model is capable of handling a wide range of downstream tasks. Below, we describe the key experimental settings, including data-selection criteria, model architectures, loss functions for the foundation model training, and the connection of reconstruction loss to mutual information. Further details of the implementation are provided in Appendix D and Appendix E.

## 2.1. Pre-trained Foundation Model for LAMOST LRS

The most recent LAMOST data release (DR11) 3 includes more than 10 million low-resolution spectra (Luo et al. 2015), used for a variety of scientific tasks, including estimation of stellar parameters, chemical abundances, reddening, radial velocity, stellar mass, and age. Our aim in pre-training a foundation model for LAMOST LRS is to learn informative and transferable representations that support many of these tasks with a small to moderate amount of labeled data.

In our pre-trained LRS model, each spectrum is tokenized into overlapping segments (tokens) of 20 flux points 20 flux points ( ≈ 22 ˚ A at a wavelength of 5000 ˚ A, corresponding to ∼ 10 resolution elements and capturing typical local line structures), with the overlap of 10 points to preserve continuity, producing 146 tokens per spectrum. These tokens, together with one special token-the logarithmic standard deviation of the spectrum-are passed through 6 self-attention lay-

2 CLIP stands for Contrastive Language-Image Pre-training. In this work, we adopt the contrastive learning concept from the original CLIP framework, applying it to spectra-spectra pre-training. Although our use case differs from the original text-image alignment, we retain the term CLIP for consistency.

3 https://www.lamost.org/dr11/

Figure 1. Architecture of SpecCLIP. Two types of spectra (Shown here are examples of normalized LAMOST LRS and Gaia XP spectra varying with metallicities) are passed through two pre-trained spectral foundation models to obtain embeddings, where the pre-trained models can be either transformer-based networks or multilayer perceptron (MLP)-based autoencoders. These embeddings are then projected into a joint embedding space, which may optionally be split into a shared and a non-shared subspace. Based on the projected embeddings, we construct various loss functions to enable CLIP-like contrastive learning, cross-modal prediction, and spectral reconstruction. The combination of these loss functions results in five model variants: a baseline CLIP without decoders, CLIP-r with only reconstruction decoders, CLIP-p with only prediction decoders, CLIP-pr with full decoders, and CLIP-split with full decoders and an explicit separation of shared and non-shared embedding spaces (see Section 2.3)

ers, each yielding a 768-dimensional token embedding. During training, 6 non-overlapping chunks (each 10 tokens, corresponding to an overall effective masking rate of ≈ 45%) are randomly masked to encourage robust representation learning. The resulting transformer-like framework with mask modeling (denoted as the masked transformer, or MT) has a total of 42.7 million trainable parameters.

## 2.2. Pre-trained Foundation Model for Gaia XP Spectra

Gaia XP low-resolution spectra (De Angeli et al. 2023), obtained via the Blue (with resolving power R ranging from 30 to 100) and Red ( R ranging from 70 to 100) Photometers (BP and RP), are essential for determining key stellar properties such as stellar atmospheric parameters and chemical compositions. The differences in resolution and wavelength range compared to LAMOST LRS motivate the construction of a separate foundation model tailored to the XP modality.

Two types of models are explored for Gaia XP. The first is a transformer-based MT model, structurally similar to that used for LAMOST LRS, but with tokenization at the individual flux-point level, yielding 343 tokens per spectrum, together with two special tokens-the mean and standard deviation of the spectrum. Like the LRS model, this version uses 6 self-attention layers and masks 6 chunks (20 tokens each, corresponding to an overall masking rate of ≈ 35%) during training.

The second model is a MLP-based ordinary autoencoder (denoted as OAE), with a bottleneck layer of 768 dimensions. Both XP models have approximately the same number of trainable parameters as the LRS model and are trained for the same number of epochs. In this paper, we use the OAE

for Gaia XP to test results in tables, unless otherwise noted, because of its better performance, as discussed in Section 6.6.

## 2.3. Contrastive Learning with Decoders

To align the two spectral modalities, we use 820,568 paired LAMOST LRS and Gaia XP spectra. The backbone of the alignment model leverages contrastive loss, augmented by auxiliary decoders that contribute to additional supervision.

As shown in Figure 1, the embeddings of the LRS and XP foundation models are projected onto a shared embedding space. The LRS projection head includes a cross-attention block with a recurrence vector that is learnable, following Parker et al. (2024). For Gaia XP, the projection head is either a cross-attention block (when following the attention-based model) or an MLP (when based on the OAE model). The core alignment objective is the contrastive loss between these projected embeddings.

To enrich the learned embeddings, we incorporate four auxiliary decoders:

- Two in-modal decoders reconstruct each spectrum from its projected embedding;
- Two cross-modal decoders predict one spectrum modality from the other.

These components serve to retain modality-specific information, support cross-modal translation, and increase the robustness of learned representations. Although our multidecoder design is motivated by the need to enrich spectral embeddings with diverse reconstruction pathways, we note that related ideas have appeared independently in other domains, such as the sensor-agnostic image retrieval framework in remote sensing proposed by Hackstein et al. (2024). As discussed in further detail in the final subsection of this section, reconstructing the original spectra also helps to increase the mutual information between the projected embeddings and the inputs.

Model Variants -To assess the contributions of each model component, we construct five SpecCLIP variants:

- CLIP: A baseline contrastive model without auxiliary decoders.
- CLIP-r: Adds only reconstruction decoders to the baseline.
- CLIP-p: Adds only cross-modal prediction decoders to the baseline.
- CLIP-pr: Adds both reconstruction and prediction decoders, implicitly encouraging shared and modalityspecific representation learning.
- CLIP-split: Extends CLIP-pr by explicitly partitioning the embedding space into shared and modalityspecific subspaces through two separate projection networks, potentially disentangling the two spaces.

## 2.4. Loss Functions during Contrastive Training

The total training objective L total for the SpecCLIP model - whether for CLIP or CLIP-variants - comprises three components: a contrastive CLIP loss, a reconstruction loss, and a cross-modal prediction loss. The total loss is a weighted sum:

<!-- formula-not-decoded -->

where δ recon and δ pred are binary indicators that control the inclusion of reconstruction losses and cross-modal prediction, respectively. The weights w recon and w pred control the relative contribution of the reconstruction and prediction losses and are fixed at 1 (i.e., equal weighting) throughout this work.

Contrastive CLIP Loss -The contrastive loss aligns the XP and LRS embeddings in a shared embedding space. Let f xp and f lrs denote the encoders for XP and LRS inputs (including their pre-trained models and projection networks). For a batch of N paired examples { ( x xp i , x lrs i ) } N i =1 , their projected embeddings are computed as described below.

In the CLIP-pr model, we use all projected embeddings and normalize them to unit length: z xp i = f xp ( x xp i ) and z lrs i = f lrs ( x lrs i ) , where each embedding is L2-normalized. In the CLIP-split model, only the shared components of the embeddings are used and similarly normalized: z xp i = f shared xp ( x xp i ) and z lrs i = f shared lrs ( x lrs i ) .

The similarity matrix are scaled by a temperature parameter τ (set to 15.5, following Parker et al. 2024):

<!-- formula-not-decoded -->

where ⟨· , ·⟩ denotes the dot product between two embedding vectors. This dot product becomes equivalent to cosine similarity if the vectors are normalized. The parameter τ controls how confidently the contrastive loss distinguishes positives from negatives. The larger τ leads to greater confidence and greater push/pull between the matched and mismatched pairs.

Then the CLIP loss is computed as:

<!-- formula-not-decoded -->

where ℓ i, : and ℓ : ,i are the similarity scores for XP-to-LRS and LRS-to-XP matching, respectively, and CE ( · , i ) denotes the cross-entropy loss with label i defined as CE ( ℓ i, : , i ) =

-log ( exp( ℓ i,i ) / ∑ N j =1 exp( ℓ i,j ) ) . The cross-entropy loss encourages the matched pairs to have a higher similarity than the mismatched ones in the shared embedding space.

Reconstruction Loss -Each modality has its own decoder to reconstruct the original spectrum from its embedding. In the CLIP-pr model, reconstruction is based on the full projected embedding. In the CLIP-r and CLIP-split models, reconstruction uses both shared and non-shared embeddings, which come from separate projection branches. Let ˆ x xp i and ˆ x lrs i denote the reconstructions. The loss is:

<!-- formula-not-decoded -->

where µ i and σ i are the mean and standard deviation of the LRS input x lrs i , used to further normalize the input before reconstruction 4 .

Cross-Modal Prediction Loss -To facilitate spectrum translation, cross-modal decoders predict one modality from the embedding of the other. In CLIP-pr, this is done by using the full embedding. In CLIP-p and CLIP-split, only the shared embedding component is used for cross-modal prediction. Let ˆ x xp ← lrs i and ˆ x lrs ← xp i be the cross-predicted spectra:

<!-- formula-not-decoded -->

## 2.5. Connection of Reconstruction Loss to Mutual Information

Minimizing the reconstruction loss encourages the latent embedding to retain as much information as possible about the original input spectra. This intuition can be formalized via the mutual information between the input spectrum (either x xp or x lrs ) and its corresponding embedding ( z xp or z lrs ). For notational simplicity, we use x and z to denote a generic input spectrum and its embedding, respectively. The mutual information can be written as (Barber &amp; Agakov 2003; Ting 2025):

<!-- formula-not-decoded -->

4 This normalization step is optional. We initially adopted this design for potential training stability. Although each spectrum is already fluxnormalized by continuum division during pre-processing, this additional normalization (1) centers the input distribution and (2) focuses the decoder on learning the shape of spectral features. This makes the projected embedding retain essential LRS information. In contrast, for Gaia XP, we reconstruct the relative fluxes (colors) normalized by the 550 nm flux, aiming to recover the full chromatic information.

where H ( x ) = -E p ( x ) [log p ( x )] is the marginal entropy of the spectrum and H ( x | z ) = -E p ( z,x ) [log p ( x | z )] is the conditional entropy given the embedding. Since H ( x ) is independent of model parameters, maximizing I ( z, x ) is equivalent to minimizing H ( x | z ) .

However, the true conditional distribution p ( x | z ) is typically intractable. In practice, we introduce a variational approximation q ( x | z ) -often realized by a neural decoder-leading to the following bound:

<!-- formula-not-decoded -->

where the inequality follows from the non-negativity of the Kullback-Leibler divergence KL( p ( x | z ) ∥ q ( x | z )) (Barber &amp; Agakov 2003; Poole et al. 2019). Thus, minimizing the expected negative log-likelihood under q ( x | z ) serves as a variational lower bound on the mutual information I ( z, x ) .

When q ( x | z ) is modeled as a Laplace distribution centered at a deterministic decoder output ˆ x ( z ) , the negative loglikelihood reduces (up to constants) to the ℓ 1 reconstruction error:

<!-- formula-not-decoded -->

which justifies the use of L1 loss in our reconstruction objective. This choice is particularly appropriate for stellar spectra, especially LAMOST LRS data, which contain sharp absorption features and may occasionally include unflagged bad pixels. The L1 loss is robust to such outliers and better preserves narrow spectral lines compared to L2.

Alternatively, assuming a Gaussian likelihood q ( x | z ) = N ( x ; ˆ x ( z ) , σ 2 I ) leads to the standard mean squared error (MSE) loss:

<!-- formula-not-decoded -->

which penalizes larger residuals more strongly and encourages smooth reconstruction. While MSE may be suitable for high-SNR or denoised spectra, it tends to overly smooth localized features and is less robust to localized artifacts.

In summary, minimizing the reconstruction loss-whether L1 or L2-amounts to maximizing a variational lower bound on the mutual information between the input spectrum and its embedding. This encourages the representation to be informative and faithful to the original input.

## 3. DOWNSTREAM TASKS

We evaluate the performance of SpecCLIP on several downstream tasks, including parameter estimation, spectral retrieval (or search), and cross-modal prediction given a query spectrum. This section focuses on parameter estimation, detailing the model choices and sample-selection criteria.

## 3.1. Models for Parameter Estimation

We explore two complementary approaches for stellar parameter estimation: MLPs and simulation-based inference (SBI), also known as implicit-likelihood or likelihood-free inference (Tejero-Cantero et al. 2020; Ho et al. 2024). SBI enables inference of parameter posteriors by learning the underlying distribution (e.g., the posterior itself) and evaluating on observed data, without requiring an explicit likelihood function.

## 3.1.1. Multilayer Perceptrons

For most downstream tasks involving LAMOST LRS and Gaia XP spectra, we employ MLPs due to their training efficiency and scalability, which make them suitable for processing large datasets with limited computational resources. Each MLP has the following layer architecture: [ input dim , 1024 , 512 , 64 , 1] , where input dim = { 1462 , 343 , 768 } depending on whether the input is raw LRS spectra, raw XP spectra, or embeddings. The output dimension is one, corresponding to a single stellar parameter. For pre-trained models via MT, we first reduce the representations by averaging over the sequence dimension (e.g., from [146 , 768] to [768] for the LAMOST model). The resulting 768-dimensional embeddings and their high-quality labels are then used as input to the MLP. For CLIP models and the pre-trained XP model via OAE, no reduction of dimension is required and we directly use the 768-dimensional (projected) embeddings together with their labels. Each MLP model has approximately 1.4 million trainable parameters when applied to the embeddings.

## 3.1.2. Simulation-Based Inference

For selected tasks, we also apply SBI for posterior inference. We report the median of the posterior as the point estimate.

We follow the Neural Posterior Estimation (NPE) framework to directly estimate the posterior distribution of parameters given observations, using neural density estimators. This NPE-based approach is relatively straightforward to implement and computationally efficient, making it a practical choice compared to other variants such as Neural Likelihood Estimation (NLE). We adopt two types of (conditional) normalizing flows as density estimators from the sbi package (Tejero-Cantero et al. 2020): Masked Autoregressive Flow (MAF) (Papamakarios et al. 2017) and Neural Spline Flow (NSF) (Durkan et al. 2019). Each SBI model uses two transformations with 60 hidden units (unless otherwise noted), totaling roughly 0.1 million trainable parameters. Details of the adopted normalizing flows are described in Appendix C.

To evaluate the calibration of the inferred posteriors, we perform simulation-based calibration (SBC, Talts et al. 2018) using rank statistics, also known as Probability Integral

Transform (PIT) values in some literature (Gneiting et al. 2007), complemented by the Kolmogorov-Smirnov (K-S) test (Kolmogorov 1992). The K-S test measures the maximum discrepancy between the empirical cumulative distribution function (CDF) of the ranks and the expected uniform CDF, providing a non-parametric test of distributional consistency. For each (out of 200 in total) spectrum-parameter pair, we draw posterior samples and compute the rank of the ground-truth parameter value. Well-calibrated posteriors yield uniformly distributed ranks, which we assess via the K-S test. We consider results valid if the p -value exceeds 0.05, indicating that there is no significant deviation from uniformity and therefore reliable posterior coverage.

## 4. DATASETS

## 4.1. Sample Selection for Pre-training

For the LRS model, we select a subset of 966,082 highquality spectra for pre-training, using a 9:1 train-validation split (the same split ratio is used for the other modeling efforts described in the following two subsections). The selection criteria are: (1) SNR in the SDSS g -band greater than 50, and (2) apparent g -band magnitude less than 15.8. Additionally, we include all spectral types beyond the AFGK classes, while attempting to balance the four AFGK types themselves. We also aim to balance giant and dwarf stars; however, due to the intrinsic distribution of the dataset, the final ratio of dwarfs to giants is approximately 3.8:1.

For the XP model, we pre-train this model using one million Gaia XP spectra, with around 80% having matching LAMOST LRS spectra. Each XP spectrum consists of 343 flux points spanning wavelengths from 336 nm to 1021 nm.

## 4.2. Sample Selection for Parameter Estimation

For LAMOST LRS spectra, we evaluate parameter estimation across several classes of physical properties:

1. Stellar atmospheric parameters: effective temperature ( T eff ), surface gravity (log g ), and iron abundance ([Fe/H]);
2. Other elemental abundances: [ α /Fe], [C/Fe], [N/Fe], [Mg/Fe], [O/Fe], [Al/Fe], [Si/Fe], [Ca/Fe], [Ti/Fe], [Mn/Fe], [Ni/Fe], [Cr/Fe];
3. Asteroseismic parameters and derived physical parameters: large frequency separation ( ∆ ν ), frequency of maximum oscillation power ( ν max ), stellar mass ( M ⊙ ), radius ( R ⊙ ), age (Gyr), and period spacing of gravity modes ( ∆Π );
4. Other parameters: radial velocity ( v r ) and extinction E ( BP -RP ) .

We selected approximately 100,000 stars per parameter to balance the parameter distribution and computational tractability. The quality-control criteria include g -band spectral SNR g &gt; 20, Gaia g -band magnitude &lt; 16.5, and | v r | &lt; 800 kms -1 to exclude likely extragalactic sources. LAMOST DR11 is used throughout; we adopt a 3 ′′ matching radius between it and other catalogs. For APOGEE, we typically (unless otherwise noted) require APOGEE SNR (median SNR per pixel in combined frame (at apStar sampling)) &gt; 50.

Sample selection strategies for individual parameters are summarized below:

- Radial Velocity ( v r ): We select 100,299 stars in common between APOGEE DR17 (Abdurro'uf et al. 2022) and the LAMOST LRS sample. Their radial velocities are approximately uniformly distributed over the range [ -800 , 800] km s -1 , divided into 800 bins, each containing up to 1600 stars.
- Effective Temperature ( T eff ): Approximately 100,000 stars are uniformly sampled in the T eff -log g plane, divided into a 100 × 100 grid, with up to four stars per bin. The values of T eff are adopted from the LAMOST catalog.
- Surface Gravity ( log g ): 100,000 stars combining log g labels from Kepler (Li et al. 2022b, mainly for red giant stars) and APOGEE DR17 (Abdurro'uf et al. 2022). Binned into 750 intervals (up to 350 stars per bin), with priority given to Kepler-based values.
- Iron Abundance ( [Fe / H] ): A total of 100,118 stars are selected from a merged dataset comprising APOGEE DR17 (Abdurro'uf et al. 2022, for stars with [Fe / H] &gt; -2 . 0 ), supplemented with metal-poor stars from the PASTEL and SAGA compilations (Huang et al. 2024), the LAMOST/Subaru VMP sample (Li et al. 2022a), and UMP datasets (Sestito et al. 2019). Atargeted sampling strategy is used to ensure balanced coverage in [Fe/H].
- Other Elemental Abundances : 85,400 stars from LAMOST-APOGEE cross-matches, filtered by APOGEE SNR &gt; 40 and valid abundance flags (i.e., C FE FLAG = 0, N FE FLAG = 0, MG FE FLAG = 0). A 2D binning in [Mg/Fe]-[Fe/H] space ( 632 × 632 bins) ensures broad and uniform sampling. The [ α /Fe] values used here are computed from APOGEE measurements as ALPHA M -( FE H -M H ), where ALPHA M includes O, Mg, Si, S, Ca, Ti, and Ti II (J¨ onsson et al. 2020).
- Extinction ( E ( BP -RP ) ): A total of 86,000 stars, with extinction values estimated using the star-pair
- technique (Yuan et al. 2013) based on the LAMOST stellar parameter catalog.
- Asteroseismic Parameters: A total of 3,029 stars have asteroseismic parameters ∆ ν and ν max derived from Kepler light curves (Li et al. 2022b; Chaplin et al. 2014), including 2,718 red giant stars and 311 mainsequence/turn-off stars. In addition, 4,034 red giant stars have measured gravity-mode period spacing ∆Π , taken from published catalogs (Vrard et al. 2016).

For Gaia XP spectra, we examine the following key parameters (with sample number in brackets) - [ α /Fe] (94,584), [C/Fe] (99,934), [N/Fe] (95,089), T eff (100,000), log g , [Fe/H] (113,218), and color excess E ( BP -RP ) (99,087) - based on consistent data sources. These datasets are crossmatched with the Gaia XP catalog. To enhance [C/Fe] coverage in the metal-poor regime, we further include stars from the LAMOST very metal-poor catalog, where [C/Fe] has been estimated using a customized version of the SEGUE Stellar Parameter Pipeline (LSSPP; Lee et al. 2015).

All datasets are divided into a ratio of 0.81:0.09:0.10 for training, validation, and testing, respectively. The validation set is used for early stopping, that is, training is halted when performance on the validation set no longer improves within 10 training epochs. The testing set is kept out and used exclusively for reporting all downstream task results shown in the tables throughout the paper. An exception is made for the figures generated from the MLP-based downstream models, where we combine the training and validation sets for model training, as explained in Section 6.5.

## 4.3. Sample Selection for External Validation

Weuse multiple external datasets for validation. For LAMOST LRS, we perform two types of comparisons:

1. Radial Velocity: Compared with GALAH DR4 (Buder et al. 2025), where 49,905 stars are selected for which GALAH's global RV fit succeeded ( FIT GLOBAL RV = True), with SNR PX CCD2 ≥ 20, LAMOST SNR g ≥ 20, and both GALAH RV COMP 1 and LAMOST RV having absolute values ≤ 999 km s -1 .

## 2. Iron Abundance:

- DESI DR1 (DESI Collaboration et al. 2025; Koposov et al. 2025): 119,335 stars are selected with LAMOST SNR g ≥ 30, DESI SN B ≥ 30, SUCCESS = True, and [Fe/H] ≥ -3 . 8 .
- GALAH DR4: 33,411 stars are retained with FLAG FE H = 0, SNR PX CCD2 ≥ 30, LAMOST SNR g ≥ 30 in both surveys, and T eff ≥ 4000 K.

For Gaia XP, the iron abundance validation uses:

1. GALAH DR4: 33,411 stars satisfying FLAG SP = 0, SNR PX CCD2 ≥ 50, and T eff ≥ 4000 K.
2. Gaia RVS catalog (Viswanathan et al. 2024): 1,413 cross-matched stars are used.

## 4.4. Preprocessing

For the LAMOST LRS spectrum, to focus on the most informative spectral features, we retain the 400-560 nm wavelength range, resulting in 1462 flux points per spectrum. The spectra are normalized before being input into the model; the normalization procedure is described in Appendix B, where an iterative polynomial fitting algorithm robustly estimates the stellar continuum across both blue and red wavelength segments while suppressing absorption features and noise. For the Gaia XP spectra, broad-band color information may sometimes carry more discriminative power than individual spectral features. We therefore normalize each spectrum by its flux at 550 nm which lies near the center of the V -band.

## 5. RESULTS

This section presents a comprehensive evaluation of SpecCLIP across multiple dimensions. We begin by comparing model variants, followed by parameter-estimation results for representative parameters using both LAMOST LRS and Gaia XP spectra. We end with demonstrations of spectral retrieval and cross-modal prediction.

## 5.1. Model Comparison

Table 1 summarizes the overall performance of different models on held-out test datasets for parameter estimation. The pre-trained model on LAMOST LRS spectra generally outperforms the raw spectra, and importantly, CLIP-based models consistently improve performance for both LAMOST LRS and Gaia XP spectra, in most tasks where raw spectra or pre-trained (on LAMOST LRS or Gaia XP only) models alone were less effective. One notable exception is the radial velocity v r from LAMOST LRS, where most CLIP-based models perform worse than the LRS pre-trained model. This is understandable, as radial velocity is primarily determined by line features in LAMOST LRS spectra, and the alignment between LAMOST and Gaia - taken at slightly different stellar epochs - may introduce inconsistencies that degrade performance. Another exception is the Asteroseismic Parameters -sbi task, where no clear differences are observed among models, possibly due to the small dataset size (3,029 stars). These results highlight the value of CLIP-based alignment.

In particular, if checked more carefully, models with inmodal reconstruction decoders (e.g., CLIP-r and CLIP-pr compared with CLIP, and CLIP-pr compared with CLIP-p) generally show improved performance, as revealed by the

'number of wins' in the table (last row), fairly accounting for the MLP-based downstream models only. This indicates enhanced informativeness of the learned representations for the estimation of downstream parameters. Overall, these results suggest that the inclusion of in-modal reconstruction decoders improves representation quality in most downstream applications.

We hypothesize that these performance gains come from the model's ability to retain shared and modality-specific (non-shared) information. In the CLIP-pr variant, this is encouraged by jointly optimizing the contrastive loss, crossmodal prediction loss, and in-modal reconstruction loss. This architecture implicitly encourages the embeddings to retain complementary information from each modality.

To further explore this hypothesis, we introduce the CLIPsplit model, which explicitly separates the projected embeddings into a 512-dimensional shared space and a 128dimensional non-shared space. Despite using fewer parameters (see Appendix E), CLIP-split performs competitively, particularly for core stellar parameters such as T eff , log g , and [Fe/H]. It also recovers radial velocity performance to a level comparable with the LAMOST LRS pre-trained model, suggesting that the embedding split scheme retains more LRSspecific line features relevant to RV estimation.

Beyond parameter estimation, we also evaluated the models on two additional tasks using 50,000 paired spectra selected from the validation split of the datasets used for contrastive training with decoders.

- Similarity Score: Measures how closely projected embeddings (or shared embeddings for CLIP-split) from different modalities align, which is crucial for cross-modal retrieval. Higher scores indicate more effective alignment.
- Cross-Modal Prediction Score: Evaluates the weighted (by measurement error 5 ) mean squared error (MSE) between predicted and ground-truth spectra in cross-modal translation (e.g., LRS → XP or XP → LRS).

These results are summarized in Table 2. We find that models with both reconstruction and prediction decoders (CLIPpr) yield improved performance on LRS → XP prediction, but slightly degrade the similarity score, an expected tradeoff when the embeddings are trained to retain both shared

5 For comparison purposes (not exactly strictly), for LAMOST LRS, we propagate the inverse variance ( ivar ) of the flux measurements by multiplying by the square of the continuum fit ( C 2 ), transforming ivar to ivar · C 2 for the normalized spectrum. For Gaia XP spectra, when normalizing the flux ( F ) by the flux at 550nm ( F 550 ), the error ( σ F N ) of the normalized flux ( F N = F/F 550 ) is calculated using the standard error propagation for division: σ F N = F N √ ( σ F /F ) 2 +( σ F 550 /F 550 ) 2 , where σ F and σ F 550 are the respective flux errors.

Table 1. Comparison of Model Performance (standard deviation of the residuals σ and coefficient of determination R 2 ) for different models evaluating on the held-out test datasets

| LRS Models                          | LRS Models                          | LRS Models                    | LRS Models                    | LRS Models                | LRS Models                    | LRS Models                    | LRS Models                    |
|-------------------------------------|-------------------------------------|-------------------------------|-------------------------------|---------------------------|-------------------------------|-------------------------------|-------------------------------|
| Parameter                           | Raw Spectra σ / R 2                 | Pre-trained σ / R 2           | CLIP σ / R 2                  | CLIP-r σ / R 2            | CLIP-p σ / R 2                | CLIP-pr σ / R 2               | CLIP-split σ / R 2            |
| Atmospheric Parameters              |                                     |                               |                               |                           |                               |                               |                               |
| [Fe / H]                            | 0.070 / - 0.882                     | 0.066 / 0.939                 | 0.058 / 0.949                 | 0.057/0.949               | 0.058 / 0.949                 | 0.057 / 0.949                 | 0.056 / 0.954                 |
| T eff (K)                           | 225.733 / 0.863                     | 147.344 / 0.989               | 131.069 / 0.990               | 137.360/ 0.990            | 131.095 / 0.990               | 128.065 / 0.990               | 132.669 / 0.990               |
| T eff -sbi (maf) (K)                | 106.903 / 0.979                     | 94.942 / 0.990                | 96.577 / 0.990                | 94.930/ 0.990             | 95.004 / 0.990                | 95.346 / 0.990                | 93.047 / 0.990                |
| T eff -sbi (nsf) (K)                | 76.986 / 0.982                      | 84.991 / 0.991 †              | 85.365 / 0.990                | 84.763/ 0.991             | 85.065 / 0.990                | 84.101 / 0.991                | 82.309 / 0.991                |
| log g                               | 0.101 / 0.958                       | 0.091 / 0.981                 | 0.086 / 0.982                 | 0.084/0.983               | 0.086 / 0.982                 | 0.085 / 0.983                 | 0.079 / 0.985                 |
| log g -sbi (maf)                    | 0.063 / 0.967                       | 0.062 / 0.981                 | 0.064 / 0.982                 | 0.065/0.983               | 0.065 / 0.984                 | 0.066 / 0.983                 | 0.064 / 0.985                 |
| Elemental Abundances                |                                     |                               |                               |                           |                               |                               |                               |
| [ α/ Fe]                            | 0.023 / 0.872                       | 0.021 / 0.906                 | 0.020 / 0.912                 | 0.020 /0.913              | 0.020 / 0.911                 | 0.020 / 0.916                 | 0.020 / 0.911                 |
| [C / Fe]                            | 0.041 / 0.758                       | 0.039 / 0.792                 | 0.037 / 0.813                 | 0.037 /0.812              | 0.037 / 0.812                 | 0.037 / 0.814                 | 0.037 / 0.813                 |
| [N / Fe]                            | 0.054 / 0.598                       | 0.052 / 0.642                 | 0.049 / 0.664                 | 0.049 /0.665              | 0.049 / 0.664                 | 0.049 / 0.667                 | 0.049 / 0.667                 |
| [Al / Fe]                           | 0.049 / 0.691                       | 0.048 / 0.711                 | 0.046 / 0.741                 | 0.046 /0.739              | 0.046 / 0.738                 | 0.046 / 0.738                 | 0.046 / 0.736                 |
| [Ca / Fe]                           | 0.032 / 0.670                       | 0.030 / 0.697                 | 0.029 / 0.719                 | 0.029 /0.720              | 0.029 / 0.721                 | 0.029 / 0.723                 | 0.029 / 0.714                 |
| [Mg / Fe]                           | 0.031 / 0.866                       | 0.032 / 0.871                 | 0.031 / 0.882                 | 0.031 /0.882              | 0.031 / 0.881                 | 0.031 / 0.883                 | 0.031 /0.880                  |
| [Si / Fe]                           | 0.029 / 0.776                       | 0.029 / 0.803                 | 0.028 / 0.813                 | 0.028 / 0.813             | 0.028 / 0.813                 | 0.028 / 0.812                 | 0.028 / 0.812                 |
| [Ti / Fe]                           | 0.061 / 0.492                       | 0.058 / 0.532                 | 0.056 / 0.550                 | 0.055 /0.551              | 0.056 / 0.551                 | 0.055 / 0.552                 | 0.056 / 0.551                 |
| [Mn / Fe]                           | 0.033 / 0.761                       | 0.032 / 0.780                 | 0.031 / 0.796                 | 0.031 / 0.798             | 0.031 / 0.797                 | 0.031 / 0.798                 | 0.031 / 0.793                 |
| [Ni / Fe]                           | 0.027 / 0.426                       | 0.026 / 0.454                 | 0.025 / 0.490                 | 0.025 /0.486              | 0.025 / 0.486                 | 0.025 / 0.488                 | 0.025 / 0.485                 |
| [O / Fe]                            | 0.051 / 0.698                       | 0.050 / 0.722                 | 0.049 / 0.729                 | 0.049/ 0.730              | 0.049 / 0.728                 | 0.048 / 0.730                 | 0.049 / 0.729                 |
| [Cr / Fe]                           | 0.081 / 0.177                       | 0.076 / 0.225                 | 0.074 / 0.242                 | 0.075/0.240               | 0.075 / 0.239                 | 0.075 / 0.239                 | 0.075 / 0.232                 |
| Asteroseismic Parameters -sbi (maf) | Asteroseismic Parameters -sbi (maf) |                               |                               |                           |                               |                               |                               |
| ∆ ν                                 | 1.372 /0.901                        | 1.705/0.958                   | 1.491/0.841                   | 1.515/0.859               | 1.630/0.818                   | 1.491/0.862                   | 1.507/ 0.963                  |
| ν max                               | 20.470 / 0.676                      | 23.597/0.330                  | 22.590/0.171                  | 23.738/0.296              | 22.149/0.615                  | 22.136/0.606                  | 23.822/0.623                  |
| Mass ( M ⊙ )                        | 0.095/0.518                         | 0.094/0.570                   | 0.086/ 0.674                  | 0.087/0.670               | 0.084 /0.669                  | 0.085/0.664                   | 0.089/0.658                   |
| Radius ( R ⊙ )                      | 0.604 /0.873                        | 0.708/0.859                   | 0.738/0.853                   | 0.728/0.853               | 0.723/0.843                   | 0.737/0.847                   | 0.713/ 0.880                  |
| Age (Gyr)                           | 1.565/0.655                         | 1.488/0.684                   | 1.397/0.721                   | 1.347/0.744               | 1.347/ 0.751                  | 1.352/0.747                   | 1.337 /0.723                  |
| ∆Π                                  | 28.078/0.891                        | 25.703/0.904                  | 21.471/0.923                  | 21.814/0.920              | 22.057/0.917                  | 21.346 /0.924                 | 22.713/ 0.925                 |
| Other Parameters                    |                                     |                               |                               |                           |                               |                               |                               |
| E ( BP - RP )                       | 0.075 / - 36.886                    | 0.076 / 0.711                 | 0.070 / 0.739                 | 0.070/0.740               | 0.069 / 0.742                 | 0.069 / 0.743                 | 0.072 / 0.741                 |
| v r (km s - 1 )                     | 6.071 / 0.970                       | 5.345 / 0.978                 | 6.782 / 0.969                 | 6.158/0.972               | 6.749 / 0.969                 | 6.243 / 0.972                 | 5.289 / 0.979                 |
| v r -sbi (maf) (km s - 1 )          | 4.573 / 0.963                       | 4.653 / 0.979                 | 5.774 / 0.959                 | 5.238/0.959               | 5.786 / 0.960                 | 5.270 / 0.961                 | 4.581 / 0.978                 |
|                                     |                                     |                               | XP Models                     |                           |                               |                               |                               |
| Parameter                           | Raw Spectra                         | Pre-trained σ / R 2           | CLIP 2                        | CLIP-r 2                  | CLIP-p 2                      | CLIP-pr 2                     | CLIP-split 2                  |
|                                     | σ / R 2                             |                               | σ / R                         | σ / R                     | σ / R                         | σ / R                         | σ / R                         |
| Atmospheric Parameters              |                                     |                               |                               |                           |                               |                               |                               |
| [Fe / H]                            | 0.469 / -0.389                      | 0.126 / 0.884                 | 0.111 / 0.900                 | 0.111 / 0.900             | 0.111 / 0.900                 | 0.112 / 0.899                 | 0.113 / 0.894                 |
| T eff (K)                           | 220.258 / 0.965                     | 199.458 / 0.969               | 172.722 / 0.974               | 169.602 / 0.974           | 172.638 / 0.974               | 171.811 / 0.974               | 170.696 / 0.973               |
| T eff -sbi (maf) (K)                | . . . / . . . ‡                     | . . . / . . .                 | 173.804 / 0.968               | . . . /. . .              | . . . / . . .                 | . . . / . . .                 | . . . / . . .                 |
| T eff -sbi (nsf) (K) log g          | 137.247 / 0.970 0.757 / 0.580       | 150.437 / 0.964 0.206 / 0.953 | 130.980 / 0.970 0.175 / 0.962 | 132.889/0.971 0.173/0.962 | 130.689 / 0.972 0.173 / 0.962 | 129.708 / 0.970 0.171 / 0.963 | 131.251 / 0.972 0.174 / 0.961 |
| log g -sbi (maf)                    |                                     |                               |                               | 0.166/                    |                               | 0.167 / 0.959                 | 0.164 / 0.959                 |
|                                     | 0.202 / 0.941                       | 0.182 / 0.952                 | 0.166 / 0.959                 | 0.959                     | 0.165 / 0.958                 |                               |                               |
| Elemental Abundances                |                                     |                               |                               |                           |                               |                               |                               |
| [ α/ Fe]                            | 0.103 / - 0.047                     | 0.056 / 0.737                 | 0.049 / 0.774                 | 0.048 / 0.777             | 0.049 / 0.773                 | 0.049 / 0.770                 | 0.050 / 0.765                 |
| [C / Fe]                            | 0.194 / 0.073                       | 0.127 / 0.527                 | 0.118 / 0.547                 | 0.118/ 0.553              | 0.118 / 0.550                 | 0.117 / 0.549                 | 0.118 / 0.551                 |
| [N / Fe]                            | 0.115 / - 4.040                     | 0.077 / 0.643                 | 0.072 / 0.673                 | 0.072 / 0.676             | 0.073 / 0.672                 | 0.072 / 0.674                 | 0.073 / 0.669                 |
| Other Parameters E ( BP - RP )      | 0.077 / 0.725                       | 0.036 / 0.921                 | 0.036 / 0.926                 | 0.035 /0.927              | 0.036 / 0.925                 | 0.035 / 0.926                 | 0.035 / 0.929                 |
| Number of wins (best σ or           | 0                                   | 0                             | 19                            | 24                        | 15                            | 29                            | 19                            |

Note. 'CLIP' for contrastive training-only model, 'CLIP-r' for CLIP+reconstruction (LRS/XP) decoders, 'CLIP-p' for CLIP+cross decoders, 'CLIP-pr' for CLIP+all decoders, 'CLIP-split' for CLIP+all decoders and an explicit separation of shared and non-shared embedding spaces. Most values are run with down-stream models of MLP but the ones with '-sbi' suffix are generated by the SBI models, where two kinds of SBI models, MAF and NSF models, are applied. Numbers in bold indicate the best performance (i.e., lowest σ or highest R 2 ) for each parameter across all models. The last row reports the number of times each model achieves the best performance (i.e., lowest σ or highest R 2 ) for any parameter, based on results from MLP-based downstream models only. Some outliers in prediction may dominate the overall R 2 , occasionally leading to negative R 2 values. The numbers are reported as the average over 5 independent training runs, except for the results using SBI. For these, we report the single best-performing run among the five, selected based on a simulation-based calibration (SBC) test with a p -value threshold of at least 0.05, and prioritized by the lowest σ . † Results marked exclude a failed NSF sampling case on one extreme spectrum. ‡ Entries with dots indicate cases where all five runs failed the SBC test and no further tuning was performed. The same convention applies to other tables in this paper. We report the robust standard deviation of residuals ( σ ) using the Tukey Biweight Scale Estimator (Hoaglin et al. 1983), implementation available in robust sigma.py , in all tables where σ was used for internal model comparisons. For the plots involving external comparisons, we instead use sigma clip from astropy with 3 σ clipping, for ease of replication.

Table 2. Comparison of Cross-Modal Prediction Errors and Similarity Scores in the Projected Embeddings

| Model      | XP → LRS MSE   | LRS → XP MSE   |   Similarity |
|------------|----------------|----------------|--------------|
| CLIP       | . . .          | . . .          |       0.7828 |
| CLIP-r     | . . .          | . . .          |       0.7783 |
| CLIP-p     | 0.3932         | 3.20 × 10 - 3  |       0.782  |
| CLIP-pr    | 0.3934         | 3.15 × 10 - 3  |       0.7778 |
| CLIP-split | 0.3929         | 3.48 × 10 - 3  |       0.7854 |

Note. Numbers in bold indicate the best performance (i.e., lowest MSE or highest similarity scores) across all models. Entries with dots indicate that the corresponding models are not applicable to the task.

and non-shared information. Nevertheless, their similarity scores remain significantly higher than the baseline similarity (0.0533) obtained from comparing the embeddings between modality-specific pre-trained models.

CLIP-split achieves the highest similarity score overall, even surpassing the baseline CLIP model, possibly aided by its lower embedding dimensionality, which tends to produce higher cosine similarities. It also delivers the best performance on the prediction of the LRS spectrum of XP → LRS, demonstrating the robustness of the model in all modalities.

In summary, models with prediction and reconstruction decoders (CLIP-pr and CLIP-split) offer the best overall performance by balancing parameter-estimation accuracy, crossmodal predictability, and embedding similarity.

## 5.2. Parameter Estimation 5.2.1. LAMOST LRS

Figure 2 presents results for radial velocity and iron abundance estimation 6 . For radial velocity, we compare predictions from the LRS pre-trained model and the CLIP-split model with the official LAMOST stellar parameter catalog, using GALAH DR4 - which is not included in the training set - as an external benchmark. While our models produce slightly larger standard deviations (4.51 and 4.53 km s -1 compared to 4.22 km s -1 from the official LAMOST pipeline), they exhibit significantly smaller biases, producing values that are closer to the true measurements.

From a computational perspective, our LRS-based RV inference is highly efficient: In an environment with 1 core (Intel® Xeon® Gold 6248 @ 2.50GHz) and 1 V100 GPU, inference takes ∼ 5ms per spectrum using SBI, and 1ms per spectrum using an MLP. However, as seen in Table 1, SBI

6 The reported std values for all plots are calculated using sigma clip from astropy with 3 σ clipping.

models provide better precision (lower σ ), while MLPs offer comparable or better accuracy ( R 2 ).

For iron abundance [Fe/H], we benchmark our models against the DESI DR1 and GALAH DR4. Compared to DESI, the CLIP-split model demonstrates significantly better performance than the LAMOST official pipeline. The plateau in [Fe/H] around -2 . 5 , which arises from the lack of metal-poor stars in the stellar library used by the LAMOST pipeline, is effectively addressed by our models, although the overall scatter remains comparable. When benchmarked against GALAH, the CLIP-split model achieves both lower bias and lower scatter, highlighting its competitive performance relative to physically motivated pipelines. We further investigated the impact of rest-frame correction by applying the predicted RVs to shift the spectra before feeding them into the [Fe/H] prediction models. This additional step yields predictions that are broadly consistent with those from uncorrected inputs, indicating that correcting for radial velocity is not critical-at least for the resolution and wavelength coverage of LAMOST LRS. The insensitivity of the MLP-based [Fe/H] predictions to small redshifts implies that the model has implicitly learned to accommodate these variations.

Although our main pipeline estimates one parameter per MLP model (multi-variate MLP is also straightforward, though we did not explore it in this paper), we also experimented with SBI variants that estimate either one or all parameters jointly. While overall performance was similar, joint estimation, especially with SBI, better captures parameter degeneracies. An example of inferred chemical abundances with SBI is shown in Appendix A, Figure A1.

In general, our method compares favorably with previous work (Xiang et al. 2019; Li &amp; Lin 2023; Wang et al. 2023; Zhang et al. 2025; Zhao et al. 2025). Compared with the other data-driven methods listed, our approach generally requires fewer labeled training samples (typically &lt; 90,000) and minimal hyperparameter tuning, as we adopt a unified architecture for all downstream models. Relative to physicsdriven methods such as template fitting or forward modeling, our model does not require synthetic spectral templates or explicit physical modeling at inference time. This design, combined with diverse training data, enables applicability across a wide range of stellar types and delivers fast predictions once trained. Although our method avoids physical modeling during inference, its effectiveness still depends on high-quality labels, all of which are ultimately derived from physics-based modeling approaches. These characteristics make our method both efficient and broadly applicable in practice. However, direct comparison with previous literature remains challenging due to differences in the composition of the test set. For example, our iron abundance test set extends to [Fe/H] ∼ -4 , increasing the difficulty of achieving a general low scatter or a high R 2 .

Figure 2. Comparison between the LAMOST catalog and SpecCLIP models (including the pre-trained LRS model and the LRS branch of the CLIP-split model). From top to bottom: The radial velocity (RV) comparison as a function of the GALAH labels; [Fe/H] comparison as a function of the DESI labels; [Fe/H] comparison as a function of the GALAH labels; and [Fe/H] comparison as a function of the GALAH labels with input spectra shifted to the rest frame using the predicted RVs from the corresponding models in the top row. For RV, which is inferred using the SBI downstream model, the pre-trained LRS model and CLIP-split model have slightly larger scatter but smaller bias, compared with the LAMOST catalog; For [Fe/H], inferred using MLP downstream models (as with all other figures), the pre-trained LRS model and CLIP-split model gives either smaller scatter over the metal-poor region (referring to DESI labels) or overall smaller scatter and bias (referring to GALAH labels). The RV-corrected spectra result in similar [Fe/H] prediction performance, suggesting that the trained MLP models are relatively robust to modest Doppler shifts in the LAMOST LRS spectra. The dashed lines are the one-to-one lines. The numbers in the upper left of each panel are the mean offsets and standard deviation of the residuals (y-axis minus x-axis).

## 5.2.2. Gaia XP

Figure 3 shows [Fe/H] predictions from the Gaia XP model (both pre-trained and CLIP-split models) compared with ground truth from the GALAH DR4 and Gaia RVS catalog. Our predictions are consistent across the entire iron abundance range, including the metal-poor regime down to about [Fe/H] = -3 . 5 or even -4 . 0 , and outperform previous machine learning methods (Andrae et al. 2023), particularly at our ability to extend to low iron abundances. The overall scatter is below 0.08 dex for stars with [Fe/H] &gt; -2 . 0 , and below 0.18 dex for stars with [Fe/H] &lt; -2 . 0 . This performance surpasses that achieved by traditional low-resolution spectroscopy.

Figure 4 highlights 135,370 extremely metal-poor (EMP) star candidates with iron abundances in the range -5 &lt; [Fe / H] &lt; -3 , identified by our XP branch of the CLIP-

Figure 3. Comparison of [Fe/H] estimates from SpecCLIP (pre-trained XP model and XP branch of the CLIP-pr model) with reference labels from GALAH (top) and Gaia RVS (bottom). Both models correlate well with reference labels, with the CLIP-pr model yielding lower scatter and bias. The dashed lines are the one-to-one lines. The numbers is the upper left of each panel are the mean offsets and standard deviation of the residuals.

pr model. These stars exhibit a pronounced concentration toward the Galactic center, reminiscent of the 'metal-poor heart of the Galaxy' reported by Rix et al. (2022), but now extending to significantly lower iron abundances than previously observed. A dedicated follow-up study based on this sample is currently underway, aiming to shed light on the earliest phases of the Milky Way's chemical and structural evolution.

Performance metrics in various XP models are shown in Table 1. Again, CLIP-based models (CLIP, CLIP-pr, CLIPsplit) are competitive compared to earlier approaches, including Huang et al. (2024) and Li et al. (2024b). Notably, our test sets span a wide parameter range. For example, [Fe/H] extends down to around -4 . 0 and T eff extends up to 13,500 K, further validating the robustness of our models.

## 5.3. Spectral Retrieval and Prediction 5.3.1. Spectral Retrieval

Beyond parameter estimation, SpecCLIP also enables retrieval of similar spectra within the learned embedding space, both within a single modality and across different modalities.

Figure 4. Spatial density distributions of extremely metal-poor stars ( -5 &lt; [Fe / H] &lt; -3 ) derived from SpecCLIP (CLIP-pr model) in Galactic coordinates, showing a clear 'metal-poor old heart' of our Galaxy.

Figure 5 shows examples of in-modal and cross-modal retrievals given a specific query spectrum. Retrieval is based on cosine similarity in the (projected) embedding space, using either the full or shared embeddings (for CLIP-split). In this figure, the search is performed using the CLIP-pr model

Figure 5. Two examples of in-modal retrieval, cross-modal retrieval, cross-modal prediction, and the LAMOST (Gaia) spectra corresponding to Gaia (LAMOST) in-modal retrieval. The similarity scores are defined in the projected embedding space.

on a test set of 82,057 spectra, with the query spectrum excluded from the database. In both LAMOST LRS and Gaia XP cases, the retrieved spectra closely resemble the query spectra, indicating that the model has learned well-aligned representations across modalities.

In practice, additional strategies can be used to retrieve spectra using auxiliary catalog links. For example, given a query LAMOST spectrum and a LAMOST-to-Gaia crossmatch library, one could first retrieve the top LAMOST matches (in-modal) and then fetch their Gaia counterparts via the library. Alternatively, if the paired Gaia spectrum of the query is known, one could retrieve similar Gaia spectra directly (in-modal), or indirectly by performing a Gaia-toLAMOST retrieval followed by a database lookup to obtain the corresponding Gaia spectra. While Figure 5 presents only two retrieval use cases, assuming that we know only the information of the query spectrum itself, not its paired othermodal spectrum. The other more elaborate approaches are straightforward extensions.

These capabilities suggest promising applications in data mining and search-based discovery. For instance, starting from a LAMOST spectrum of a rare type of star, one could search for spectrally similar candidates in the Gaia XP database of over two billion stars. Such functionality could

Table 3. Comparison of Model Performance (standard deviation of the residuals σ and coefficient of determination R 2 ) for pre-trained XP models with different embedding dimensions (256, 343, 512, 768), where 768 is adopted in this paper

| XP Models              | XP Models              | XP Models              | XP Models              | XP Models              | XP Models              |
|------------------------|------------------------|------------------------|------------------------|------------------------|------------------------|
| Parameter              | Raw Spectra σ / R 2    | 256 σ / R 2            | 343 σ / R 2            | 512 σ / R 2            | 768 σ / R 2            |
| Atmospheric Parameters | Atmospheric Parameters | Atmospheric Parameters | Atmospheric Parameters | Atmospheric Parameters | Atmospheric Parameters |
| [Fe / H]               | 0.469 / - 0.389        | 0.128 / 0.881          | 0.127 / 0.882          | 0.129 / 0.879          | 0.126 / 0.884          |
| T eff (K)              | 220.258 / 0.965        | 201.126 / 0.967        | 203.142 / 0.967        | 196.401 / 0.968        | 199.458 / 0.969        |
| log g                  | 0.757 / 0.580          | 0.207 / 0.952          | 0.208 / 0.951          | 0.197 / 0.955          | 0.206 / 0.953          |
| Elemental Abundances   | Elemental Abundances   | Elemental Abundances   | Elemental Abundances   | Elemental Abundances   | Elemental Abundances   |
| [ α/ Fe]               | 0.103 / - 0.047        | 0.057 / 0.732          | 0.057 / 0.731          | 0.056 / 0.738          | 0.056 / 0.737          |
| [C / Fe]               | 0.194 / 0.073          | 0.129 / 0.527          | 0.128 / 0.525          | 0.129 / 0.526          | 0.127 / 0.527          |
| [N / Fe]               | 0.115 / - 4.040        | 0.079 / 0.631          | 0.080 / 0.628          | 0.077 / 0.641          | 0.077 / 0.643          |
| Other Parameters       | Other Parameters       | Other Parameters       | Other Parameters       | Other Parameters       | Other Parameters       |
| E ( BP - RP )          | 0.077 / 0.725          | 0.038 / 0.913          | 0.039 / 0.915          | 0.036 / 0.915          | 0.036 / 0.921          |

Note. Numbers in bold indicate the best performance (i.e., lowest σ or highest R 2 ) for each parameter across all models.

significantly enhance large-scale searches for rare or unusual stellar types.

## 5.3.2. Spectral Prediction

SpecCLIP's cross-modal decoders also support spectral translation, that is, predicting the spectrum in one modality from a spectrum in another. These decoders operate directly on the projected embeddings (shared embeddings in the case of CLIP-split), using learned mappings between modalities.

We find that for the majority of the test dataset, the model performs well in both directions (LRS → XP and XP → LRS), indicating that it effectively learns the mapping between these two spectroscopic modalities. Examples of such predictions are shown in Figure 5 as black curves.

However, for a subset of sources-particularly in the LRS → XP direction-prediction quality deteriorates. Apart from the potential effect of extinction, which may be poorlycaptured by the trained model. This discrepancy may suggest that the source does not follow the behavior of a typical single star. For instance, it could be an unresolved binary or an otherwise anomalous object. These cases highlight an exciting future direction, using the cross-modal prediction error as a basis for anomaly detection.

We leave a more systematic exploration of anomaly detection and rare-object identification to future work.

## 6. DISCUSSION

## 6.1. SBI Performance with NSF and MAF

In Table 1, we present the SBI results for several parameters ( T eff , log g , v r , and asteroseismic parameters) that demonstrate the benefits of adopting SBI. For parameters showing relatively clear differences between NSF and MAF, we include results from both; for those exhibiting visually biased predictions in one of the two SBI methods, we conservatively report only the other. For parameters modeled by both methods (e.g., T eff ), we observe better performance with NSF, likely due to its higher flexibility in representing non- linear conditional dependencies between spectra and stellar parameters.

In our additional experiments, we observed that applying NSF to projected embeddings led to degraded performance in the low-iron abundance regime ( [Fe / H] ≲ -1 . 5 ). Similarly, LAMOST LRS radial velocity predictions exhibited a systematic underestimation of the absolute value at large radial velocities; a similar issue is widely discussed in machine learning-based estimators (Ting 2024). Interestingly, replacing NSF with MAF eliminated these issues.

These results suggest that NSF is more expressive than MAF, this increased expressiveness can adversely affect performance in our applications when modeling distributions near parameter-space boundaries. A likely explanation is that the spline transformations used in NSF, while highly flexible within their domain, become problematic in sparsely sampled boundary regions. Specifically, NSF employs rational-quadratic splines over a finite interval (default: ± 3 σ in standardized space), with linear tail transformations beyond this range.

In astrophysical parameter spaces where extreme values ( [Fe / H] ≲ -1 . 5 , | v rad | ≳ 150 km s -1 ) represent several percent of the population, these linear tails systematically mis-model the true distribution shape. Furthermore, the flexibility of spline binning can lead to overfitting in regions with sparse training coverage, where bin allocation becomes poorly constrained.

In contrast, MAF's simpler affine transformations naturally extend over the full unbounded support of the distribution with stable linear extrapolation, providing more robust behavior at distribution boundaries despite lower overall expressiveness. The contrasting performance between NSF and MAF highlights the importance of matching normalizing flow architecture to the characteristics of astrophysical parameter distributions, and motivates further investigation of boundary-aware flow designs for simulation-based inference in astronomy.

## 6.2. Compression vs. Feature Learning

A common assumption is that compression improves downstream performance. However, our Gaia XP foundation model shows a more nuanced behavior. In Table 3, we vary the embedding dimension and find that reducing it from 343 (the original XP spectrum length) to 256 leads to comparable performance. In contrast, using embedding dimensions equal to or greater than the input size yields better results.

This suggests that effective feature learning. rather than compression alone, is key to high downstream performance in this case. Mapping 343 flux points into a higherdimensional latent space (e.g., 768) may allow the model to capture richer nonlinear correlations and disentangle latent physical factors (e.g., T eff , log g , [Fe/H], extinction), forming expressive embeddings for downstream prediction. We note, however, that increasing the embedding dimension also increases the number of trainable parameters in the downstream MLP, which may partly explain the improved performance. These interpretations remain tentative and will be further examined in future work.

## 6.3. Interpretability of Parameter Estimation

One potential concern is that the models may rely on spurious correlations in the data to estimate parameters. While forward models like Payne and DD-Payne allow straightforward inspection of such behavior, our neural networks do not offer easy interpretability.

Recent techniques, such as sparse autoencoders (Cunningham et al. 2023), could improve the explainability of the model in future work. However, our model achieves high precision and accuracy on held-out datasets, suggesting that it is indeed learning physically meaningful representations. Further testing with carefully designed datasets will be necessary to validate this assumption and improve model interpretability.

## 6.4. SBI vs. MLP

Although we report MLP results for most tasks, we observe that SBI tends to outperform MLP in terms of uncertainty (as measured by σ ), while MLP yields comparable or better accuracy (as measured by R 2 ), as shown in Table 1.

This discrepancy may arise from their differing training objectives; MLPs minimize the MSE, which aligns closely with R 2 , whereas SBI focuses on modeling the entire posterior distribution. In this work, we estimate the posterior median from SBI (rather than the mean), which is more robust to outliers. However, we note that differences in the adopted hyperparameters (see Appendix G) may also contribute to the discrepancy

Thus, for applications where uncertainty quantification is essential, SBI is the preferred choice. For fast and accurate point estimates, MLP is more suitable. In particular, our

SBI models use only ∼ 0.1 million trainable parameters, compared to ∼ 1.4 million for the MLP models.

## 6.5. Training Sample Size and Dataset Configuration

We observe a positive correlation between the size of the training set and the performance of the model in the downstream tasks. Although our benchmark experiments use ∼ 100,000 stars per parameter, we note that performance may not have plateaued, indicating room for improvement with additional data.

Nevertheless, even with this moderate sample size, our models match or outperform state-of-the-art results in the literature (Section 5.2). To maximize data usage, we report metrics based on held-out test sets, while for plots generated using the downstream models, we combine the training and validation sets for training, with early stopping based on performance on the test set.

Future work may adopt more advanced techniques such as k -fold cross-validation, which would allow iterative use of the entire dataset and further improve model reliability and performance.

## 6.6. Is Transformer Overkill?

In Table 4, we compare MT and OAE with equivalent numbers of trainable parameters and identical training epochs. For LAMOST LRS spectra, MT outperforms OAE. However, for Gaia XP spectra, OAE performs better.

These results suggest that transformers may be more advantageous for longer spectra (e.g., LRS with 1462 flux points), while offering limited benefits, or even unnecessary complexity, for shorter inputs like Gaia XP spectra (343 points). Another possible explanation is the difference in optimal training epochs for different architectures, as discussed in R´ o˙ za´ nski &amp; Ting (2025) and R´ o˙ za´ nski et al. (2025). This warrants further exploration to better match model capacity to data complexity.

## 6.7. Fine-tuning Loss Weights

Not all non-shared information in the spectra is necessarily beneficial-some components, such as noise, may even hinder downstream performance. Moreover, the importance of shared versus non-shared information can vary across tasks; some may depend more on modality-specific features, while others benefit primarily from shared representations. Therefore, fine-tuning the weight terms in Equation 1, particularly the loss weight of reconstruction, may be a promising direction for future work.

In this work, we fix both weights to 1, applying equal weighting to the reconstruction and cross-modal prediction losses. This choice shows that models incorporating reconstruction loss already perform competitively without explicit weight adjustment, as evidenced by the number of highest

performing metrics (i.e. 'wins') in Table 1 (last row)-e.g., CLIP-r vs. CLIP, and CLIP-pr/CLIP-split vs. CLIP-p. However, the magnitude of performance gains remains modest and somewhat task-dependent. Reweighting the losses may further increase the total number of improvements or yield more substantial gains on specific parameters. However, the latter-optimizing for specific tasks-might come at the cost of generality, which is contrary to the fundamental goal of building a model that performs robustly across diverse downstream tasks.

Table 4. Comparison of Model Performance (standard deviation of the residuals σ and coefficient of determination R 2 ) between masked transformer (MT) and MLP-based ordinary auto-encoder (OAE)

| LRS Models                     | LRS Models             | LRS Models             | LRS Models             |
|--------------------------------|------------------------|------------------------|------------------------|
| Parameter                      | Raw Spectra σ / R 2    | MT σ / R 2             | OAE σ / R 2            |
| Atmospheric Parameters         | Atmospheric Parameters | Atmospheric Parameters | Atmospheric Parameters |
| [Fe / H]                       | 0.070 / - 0.882        | 0.066 / 0.939          | 0.070 / 0.905          |
| T eff (K)                      | 225.733 / 0.863        | 147.344 / 0.989        | 181.777 / 0.975        |
| log g                          | 0.101 / 0.958          | 0.091 / 0.981          | 0.084 / 0.973          |
| Elemental Abundances           | Elemental Abundances   | Elemental Abundances   | Elemental Abundances   |
| [ α/ Fe]                       | 0.023 / 0.872          | 0.021 / 0.906          | 0.020 / 0.904          |
| [C / Fe]                       | 0.041 / 0.758          | 0.039 / 0.792          | 0.039 / 0.776          |
| [N / Fe]                       | 0.054 / 0.598          | 0.052 / 0.642          | 0.053 / 0.624          |
| [Al / Fe]                      | 0.049 / 0.691          | 0.048 / 0.711          | 0.049 / 0.693          |
| [Ca / Fe]                      | 0.032 / 0.670          | 0.030 / 0.697          | 0.031 / 0.688          |
| [Mg / Fe]                      | 0.031 / 0.866          | 0.032 / 0.871          | 0.030 / 0.873          |
| [Si / Fe]                      | 0.029 / 0.776          | 0.029 / 0.803          | 0.029 / 0.793          |
| [Ti / Fe]                      | 0.061 / 0.492          | 0.058 / 0.532          | 0.059 / 0.507          |
| [Mn / Fe]                      | 0.033 / 0.761          | 0.032 / 0.780          | 0.033 / 0.758          |
| [Ni / Fe]                      | 0.027 / 0.426          | 0.026 / 0.454          | 0.027 / 0.445          |
| [O / Fe]                       | 0.051 / 0.698          | 0.050 / 0.722          | 0.051 / 0.704          |
| [Cr / Fe]                      | 0.081 / 0.177          | 0.076 / 0.225          | 0.079 / 0.200          |
| Other Parameters E ( BP - RP ) | 0.076 / - 23.199       | 0.076 / 0.711          | 0.076 / 0.681          |
| v r (km s - 1 )                | 6.418 / 0.942          | 5.345 / 0.978          | 5.938/0.966            |
| XP Models                      | XP Models              | XP Models              | XP Models              |
| Parameter                      | Raw Spectra σ / R 2    | MT σ / R 2             | OAE σ / R 2            |
| Atmospheric Parameters         | Atmospheric Parameters | Atmospheric Parameters | Atmospheric Parameters |
| [Fe / H]                       | 0.469 / - 0.389        | 0.137 / 0.867          | 0.126 / 0.884          |
| T eff (K)                      | 220.258 / 0.965        | 215.299 / 0.965        | 199.458 / 0.969        |
| log g                          | 0.757 / 0.580          | 0.206 / 0.953          | 0.206 / 0.953          |
| Elemental Abundances           | Elemental Abundances   | Elemental Abundances   | Elemental Abundances   |
| [ α/Fe ]                       | 0.103 / - 0.047        | 0.059 / 0.713          | 0.056 / 0.737          |
| [C / Fe]                       | 0.194 / 0.073          | 0.132 / 0.498          | 0.127 / 0.527          |
| [N / Fe]                       | 0.115 / - 4.040        | 0.079 / 0.615          | 0.077 / 0.643          |
| Other Parameters E ( BP - RP ) | 0.077 / 0.725          | 0.041 / 0.913          | 0.036 / 0.921          |

Note. Numbers in bold indicate the best performance (i.e., lowest σ or highest R 2 ) for each parameter across all models.

In this work, we develop a foundation model framework for stellar spectra that enables strong and efficient performance across multiple downstream tasks. Our approach integrates separate pre-trained models, each trained on a distinct spectroscopic modality (LAMOST LRS or Gaia XP), and aligns them using CLIP-style contrastive learning. To further enhance the information capacity of the embeddings, we introduce decoder modules that increase the mutual information between the embeddings and input spectra and enable translation (prediction) between different spectral types.

Our main findings are summarized below:

- The pre-trained foundation models for both spectral modalities demonstrate strong performance with a relatively small number of labeled examples (i.e., fewshot learning). Using ∼ 100,000 stars with highquality labels, they achieve competitive parameter inference performance across a range of stellar parameters. Comparisons with the LAMOST official release and high-resolution reference catalogs (e.g., GALAH, and APOGEE) confirm the accuracy and reliability of our method.
- Performance is further improved by contrastive alignment and the addition of decoders, which increase the robustness and expressiveness of the learned embeddings. These enhancements are especially beneficial for parameter estimation and spectral prediction.
- We explore the use of SBI as an alternative to MLPs for downstream parameter estimation. SBI provides improved uncertainty modeling and higher precision for certain parameters, albeit at a higher inference cost and a lower model capacity.
- Our models support both in-modal and cross-modal spectrum retrieval, as well as spectrum-to-spectrum prediction across modalities. High similarity and prediction scores demonstrate that the learned representations capture shared physical information. These modules also offer promising avenues for anomaly detection and similarity-based searches in large spectral archives.

Looking ahead, we plan to extend this framework to additional spectroscopic modalities, including LAMOST medium-resolution spectra (MRS, Li et al. 2024a), APOGEE infrared spectra (Majewski et al. 2017), Subaru PFS spectra (Takada et al. 2014), and DESI DR1 spectra (DESI Collaboration et al. 2025). Our approach can be readily adapted to new instruments by pre-training modality-specific encoders and aligning them with contrastive objectives and decoder structures developed in this work. In future iterations, we will also explore efficient adaptation via neural network adapters,

enabling scalable multi-survey alignment with minimal computational cost. In forthcoming work, a large-scale application and catalog release is planned.

## ACKNOWLEDGEMENTS

The Guoshoujing Telescope (the Large Sky Area MultiObject Fiber Spectroscopic Telescope, LAMOST) is a National Major Scientific Project built by the Chinese Academy of Sciences. Funding for the project has been provided by the National Development and Reform Commission. LAMOST is operated and managed by the National Astronomical Observatories, Chinese Academy of Sciences.

CEE), and OISE-1927130; The International Research Network for Nuclear Astrophysics (IReNA), awarded by the US National Science Foundation, and DE-SC002312; the Center for Nuclear Astrophysics Across Messengers (CeNAM), awarded by the U.S. Department of Energy, Office of Science, Office of Nuclear Physics. Y.S.T. is supported by the National Science Foundation under Grant No. AST2406729. G.X. and X.T. acknowledge the support from Key R&amp;D Program of Zhejiang (2024SSYS0006).

We are thankful for useful discussions with Ce Sui, Alexander S. Szalay, Rosemary F.G. Wyse, and Benjamin D. Wandelt during the early stages of this work.

Y.H. acknowledges support from the National Science Foundation of China (NSFC grant No. 12422303), the Fundamental Research Funds for the Central Universities (grant Nos. 118900M122, E5EQ3301X2, and E4EQ3301X2), and the National Key R&amp;D Program of China (grant No. 2023YFA1608303). X.Z. acknowledges partial support through a grant from the Schmidt Sciences Foundation. While this project was initiated prior to his current appointment at JHU, a significant portion of the work was completed during his Schmidt-supported position. T.C.B. acknowledges partial support from grants PHY 14-30152; Physics Frontier Center/JINA Center for the Evolution of the Elements (JINA-

## DATA AVAILABILITY

All observational data used in this work are publicly available from the archives. A frozen version of the SpecCLIP software used in this analysis has been archived on Zenodo (Zhao &amp; Huang 2025) in compliance with the AAS Journals software policy. No proprietary data were used.

Software: SpecCLIP (Zhao &amp; Huang 2025), PyTorch (Paszke et al. 2019), Astropy (Astropy Collaboration et al. 2013, 2018)

## APPENDIX

Here we present additional information on multiple aspects of this work, including elemental-abundance prediction, continuum fitting, normalization flows for parameter estimation, the use of pre-trained models, projection models and decoders, and loss curves.

## A. ADDITIONAL RESULTS FOR ELEMENTAL-ABUNDANCE ESTIMATION

Figure A1 presents an example of chemical-abundance predictions from one input spectrum. This result is obtained using SBI trained with a single model for all parameters simultaneously. The inset in the upper right corner shows results from simulationbased calibration (SBC). Most elemental abundances exhibit well-calibrated posteriors, except for [Ti/Fe] and [Ca/Fe], which fall below the significance threshold of 0.05. This provides an example where, despite the model learns posterior distributions well overall, further tuning of the SBI architecture or its hyper-parameters is necessary to achieve reliable multivariate inference of elemental abundances-an aspect we did not further refine in this work.

In Table A1, we provide downstream task results using CLIP-based models aligned between the LAMOST LRS MT and Gaia XP MT encoders. This serves as a comparison to Table 1, where alignment is performed between the LAMOST LRS MT and Gaia XP OAE models. We find that the latter configuration shows overall comparable or better performance across the evaluation metrics, possibly due to the stronger pretraining of the XP OAE model. For completeness, in Figure A2 we include an external comparison analogous to Figure 2 in the main text, but using the models from Table A1.

## B. CONTINUUM FITTING ALGORITHM

Although we applied continuum fitting only to the blue segment ( 4000 ˚ A ≤ λ ≤ 5600 ˚ A) of the LAMOST LRS spectra for our analysis, we describe here the full continuum fitting algorithm, which is designed to robustly estimate the stellar continuum over the full wavelength range ( 3850 ˚ A ≤ λ ≤ 9000 ˚ A), and remains effective under varying SNRs. The method takes as input the observed wavelength array w , the corresponding flux array f , and an estimate of the average SNR, and returns a smooth continuum model c .

The procedure is summarized as follows:

Figure A1. An example of posterior distributions of 12 elemental abundances inferred using simulation-based inference (SBI), with a downstream model trained jointly in the 12-dimensional parameter space. The embeddings used for training are from the pre-trained LRS foundation model. The upper-right panel shows simulation-based calibration (SBC) results, indicating that most posteriors are well-calibrated except for [Ti/Fe] and [Ca/Fe], which fall below the 0.05 significance threshold.

1. Pre-processing: The flux array is first smoothed using a median filter of width 7 pixels to reduce the impact of narrow-line features and noise. The resulting smoothed flux is split into two wavelength segments: a blue side ( 3700 ≤ λ ≤ 5700 ˚ A) and a red side ( λ &gt; 6100 ˚ A).
2. Denoising with Savitzky-Golay filter: The Savitzky-Golay filter is applied to each segment independently, with a smoothing window size of 3.
3. Blue Segment Adjustment (if sufficiently sampled): A fifth-order polynomial is initially fit to the smoothed blue-side flux to identify the peak region. If the peak occurs at wavelengths &lt; 4500 ˚ A, interpolation is applied over three manu-

Table A1. Similar to Table 1 for 'LRS Models', but the results for all CLIP-based models are from the alignment between the LAMOST LRS MT and Gaia XP MT (see Section 6.6 for discussion, and Appendices D and E for model details).

| LRS Models                | LRS Models          | LRS Models          | LRS Models      | LRS Models      | LRS Models      | LRS Models         |
|---------------------------|---------------------|---------------------|-----------------|-----------------|-----------------|--------------------|
| Parameter                 | Raw Spectra σ / R 2 | Pre-trained σ / R 2 | CLIP σ / R 2    | CLIP-p σ / R 2  | CLIP-pr σ / R 2 | CLIP-split σ / R 2 |
| Atmospheric Parameters    |                     |                     |                 |                 |                 |                    |
| [Fe / H]                  | 0.070 / - 0.882     | 0.066 / 0.939       | 0.058 / 0.948   | 0.059 / 0.948   | 0.057 / 0.949   | 0.058 / 0.950      |
| T eff (K)                 | 225.733 / 0.863     | 147.344 / 0.989     | 139.242 / 0.989 | 129.569 / 0.990 | 133.980 / 0.990 | 131.461 / 0.990    |
| T eff -sbi (maf) (K)      | 106.903 / 0.979     | 94.942 / 0.990      | 97.396 / 0.990  | 95.147 / 0.989  | 96.208 / 0.990  | 93.155 / 0.990     |
| T eff -sbi (nsf) (K)      | 76.986 / 0.982      | 84.991 / 0.991 †    | 86.515 / 0.991  | 86.133 / 0.989  | 84.517 / 0.990  | 84.694 / 0.992     |
| log g                     | 0.101 / 0.958       | 0.091 / 0.981       | 0.087 / 0.982   | 0.084 / 0.982   | 0.084 / 0.982   | 0.083 / 0.983      |
| log g -sbi(maf)           | 0.063 / 0.967       | 0.062 / 0.981       | 0.064 / 0.985   | 0.065 / 0.983   | 0.064 / 0.983   | 0.064 / 0.983      |
| Elemental Abundances      |                     |                     |                 |                 |                 |                    |
| [ α/ Fe]                  | 0.023 / 0.872       | 0.021 / 0.906       | 0.020 / 0.912   | 0.020 / 0.912   | 0.020 / 0.913   | 0.021 / 0.909      |
| [C / Fe]                  | 0.041 / 0.758       | 0.039 / 0.792       | 0.038 / 0.806   | 0.038 / 0.804   | 0.038 / 0.805   | 0.038 / 0.803      |
| [N / Fe]                  | 0.054 / 0.598       | 0.052 / 0.642       | 0.050 / 0.662   | 0.050 / 0.661   | 0.050 / 0.665   | 0.051 / 0.657      |
| [Al / Fe]                 | 0.049 / 0.691       | 0.048 / 0.711       | 0.046 / 0.739   | 0.046 / 0.739   | 0.046 / 0.740   | 0.046 / 0.737      |
| [Ca / Fe]                 | 0.032 / 0.670       | 0.030 / 0.697       | 0.029 / 0.714   | 0.029 / 0.714   | 0.029 / 0.716   | 0.030 / 0.711      |
| [Mg / Fe]                 | 0.031 / 0.866       | 0.032 / 0.871       | 0.031 / 0.882   | 0.031 / 0.882   | 0.031 / 0.881   | 0.031 / 0.878      |
| [Si / Fe]                 | 0.029 / 0.776       | 0.029 / 0.803       | 0.028 / 0.814   | 0.028 / 0.813   | 0.028 / 0.816   | 0.028 / 0.807      |
| [Ti / Fe]                 | 0.061 / 0.492       | 0.058 / 0.532       | 0.056 / 0.551   | 0.056 / 0.552   | 0.056 / 0.555   | 0.056 / 0.544      |
| [Mn / Fe]                 | 0.033 / 0.761       | 0.032 / 0.780       | 0.031 / 0.800   | 0.031 / 0.799   | 0.031 / 0.798   | 0.031 / 0.792      |
| [Ni / Fe]                 | 0.027 / 0.426       | 0.026 / 0.454       | 0.025 / 0.487   | 0.025 / 0.489   | 0.025 / 0.489   | 0.026 / 0.479      |
| [O / Fe]                  | 0.051 / 0.698       | 0.050 / 0.722       | 0.049 / 0.728   | 0.049 / 0.729   | 0.049 / 0.729   | 0.049 / 0.728      |
| [Cr / Fe]                 | 0.081 / 0.177       | 0.076 / 0.225       | 0.075 / 0.234   | 0.075 / 0.233   | 0.075 / 0.237   | 0.075 / 0.232      |
| Other Parameters          |                     |                     |                 |                 |                 |                    |
| E ( BP - RP )             | 0.075 / - 36.886    | 0.076 / 0.711       | 0.070 / 0.746   | 0.069 / 0.748   | 0.069 / 0.748   | 0.070 / 0.742      |
| v r (km s - 1 )           | 6.071 / 0.970       | 5.345 / 0.978       | 6.785 / 0.969   | 6.834 / 0.969   | 6.226 / 0.973   | 5.361 / 0.979      |
| v r -sbi(maf) (km s - 1 ) | 4.573 / 0.963       | 4.653 / 0.979       | 5.636 / 0.958   | 5.621 / 0.963   | 5.070 / 0.965   | 4.578 / 0.981      |

Note. Numbers in bold indicate the best performance (i.e., lowest σ or highest R 2 ) for each parameter across all models. † Results marked exclude a failed NSF sampling case on one extreme spectrum.

ally selected continuum windows ([4030-4160], [4270-4410], [4800-4940] ˚ A) to estimate local maxima and reduce the influence of absorption features on the continuum estimation. An iterative process is then performed. In each iteration, a fifth-order polynomial is fit to the updated flux, and the fit is used to suppress absorption features, effectively lifting the continuum. Ten such iterations are performed to ensure convergence.

4. Red Segment Correction: A fourth-order polynomial fit is applied iteratively to the red-side segment. At each step, outliers deviating more than 3 σ below the fit (or any point below the fit) are replaced by the polynomial value. This process is repeated up to 8 iterations to suppress absorption features and stabilize the continuum estimate.
5. Final Continuum Assembly: The fitted continuum segments are combined to form the full continuum model c over the input wavelength grid. Values of c ≤ 0 are replaced by 1.0 to ensure a strictly positive continuum.

## C. NORMALIZING FLOWS FOR PARAMETER ESTIMATION

Normalizing flows provides a flexible and tractable approach to modeling complex probability distributions by transforming a simple base distribution through a sequence of invertible and differentiable mappings. In SBI, we use normalizing flows to learn an approximate posterior q ϕ ( θ | x ) , conditioned on observations x , following the Neural Posterior Estimation (NPE) framework.

Let z ∼ p Z ( z ) denote a sample from a base distribution (e.g., standard Gaussian), and let f ϕ ( · ; x ) be an invertible transformation conditioned on x , mapping z ↦→ θ . Then, the density of θ under the flow model is given by the change-of-variables formula:

<!-- formula-not-decoded -->

where θ = f ϕ ( z ; x ) .

For simplicity, we present the flow as a single transformation f ϕ , but in practice it consists of a sequence of transformations:

<!-- formula-not-decoded -->

Figure A2. Similar to Figure 2, but the results for all CLIP-based models are from the alignment between the LAMOST LRS MT and Gaia XP MT (see Section 6.6 for discussion, and Appendices D and E for model details).

where each f k is an invertible and differentiable function with a tractable Jacobian. The log-determinant of the full transformation is the sum of the log-determinants of each layer:

<!-- formula-not-decoded -->

where h k = f k ( h k -1 ) , with h 0 = z and h K = θ .

## C.1. Masked Autoregressive Flow (MAF)

MAF (Papamakarios et al. 2017)) models the forward transformation z ↦→ θ as a sequence of autoregressive operations. Each component of θ is computed as:

<!-- formula-not-decoded -->

where µ i and σ i are outputs of neural networks that depend on the previous components z &lt;i (or equivalently, the previous output components θ &lt;i ) and the conditioning variable x . The Jacobian of this transformation is lower triangular, which allows the

log-determinant to be computed efficiently:

<!-- formula-not-decoded -->

C.2. Neural Spline Flow (NSF)

NSF (Durkan et al. 2019) generalizes MAF by replacing affine transformations with monotonic rational-quadratic splines. Each component is transformed as:

<!-- formula-not-decoded -->

where ψ i are spline parameters (bin widths, heights, and derivatives) predicted by a neural network conditioned on z &lt;i and x . The log-determinant of the Jacobian is given by:

<!-- formula-not-decoded -->

where each derivative term is efficiently computed from the analytical form of the spline. Note that while this autoregressive form is analogous to MAF, the default implementation based on the provided code is often the coupling layer variant of NSF (NSF-C), which is typically more efficient for density evaluation. Details on the coupling layer structure can be referred to in Durkan et al. (2019).

In summary, both MAF and NSF enable flexible posterior approximation in the NPE setting, while maintaining exact likelihood evaluation and efficient training via maximum likelihood.

## D. PRE-TRAINED MODELS

In this paper, we tried two different kinds of pre-trained models, one is the transformer-based model and the other is a MLPbased auto-encoder. Both kinds of networks have the same number of trainable parameters (42.7 million). We use a batch size of 64 per GPU. The models are optimized using AdamW (Loshchilov &amp; Hutter 2017) with a learning rate of 1 × 10 -5 and a weight decay of 0.1. The learning rate follows a cosine annealing schedule with linear warm-up.

## D.1. Transformer-Based Spectral Pre-trained Models

We describe two transformer models designed for the masked reconstruction of stellar spectra, based on Parker et al. (2024), each tailored to the characteristics of a different input data set: LAMOST and Gaia XP.

## D.1.1. Masked Spectral Modeling for LAMOST Spectra

This model is designed for higher-resolution (compared with Gaia XP) spectra from instruments like LAMOST, where each input sample is x ∈ R T × 1 with T = 1462 wavelength bins. The model operates as a masked sequence autoencoder using transformers.

Input Pre-processing. -Each spectrum x ∈ R T × 1 is standardized with:

<!-- formula-not-decoded -->

Then, the standardized spectrum is sliced into overlapping chunks of length L (e.g., L = 20 ) with overlap O = 10 , forming an input sequence of length S = 146 , and a special token x ′ 0 = log 10 σ is pre-pended, forming an extended sequence x ′ ∈ R T ′ × ( L +1) with sequence length T ′ = S +1 .

Model Architecture. -The input is linearly embedded and added to the learned positional embeddings. Then it passes through N = 6 layers of standard transformer blocks with H = 6 attention heads. The output is normalized and decoded through a linear projection.

Masking Strategy. -To train the model in a self-supervised fashion, we applied a chunk-based masking strategy. Given an input sequence of length T ′ , we conceptually divide it into M = 6 segments of equal-length and randomly select a contiguous piece of width w = 10 within each segment to mask. For each chunk, its starting index is sampled uniformly from the allowable range within the segment. This ensures that the masked regions are distributed across the sequence and sufficiently separated.

Formally, let the i -th segment span the indices [ s i , s i + ℓ ] , where ℓ = ⌊ T ′ /M ⌋ . Then the masked region for segment i is:

<!-- formula-not-decoded -->

<!-- formula-not-decoded -->

where t ∈ [0 , T ′ -1] is the sequence index. This strategy preserves long-range contextual integrity and forces the model to interpolate realistic spectral values across variable scales. Compared to random masking, chunk-based masking is more appropriate for spectral data, where features span contiguous wavelength regions.

Loss Function. -Let f θ (˜ x ) denote the model's reconstruction output, the model is trained to reconstruct only the masked regions using a masked MSE loss:

<!-- formula-not-decoded -->

where m t = 1 if t is masked and 0 otherwise.

LAMOST spectra benefit from local patterns and detailed features. Thus, using dense chunk-based masking and slicing helps the transformer leverage locality while preserving the global context. We train this model with 8 GPUs over a total of 128 epochs, a process that takes roughly 20 hours.

<!-- formula-not-decoded -->

This model targets low-resolution Gaia XP spectra where each sample is x ∈ R T × 1 with T = 343 wavelength bins.

Input Pre-processing. -As in the LAMOST model, we standardize the spectrum and pre-pend two (mean and standard deviation) tokens:

<!-- formula-not-decoded -->

The masked input ˜ x is defined as:

forming an extended sequence x ′ ∈ R ( T +2) × 1 .

The model architecture and the loss function are similar to the LAMOST case.

We train this model with 8 GPUs and a total of 191 epochs, which is roughly 40 hours.

## D.2. MLP-based Autoencoder

For comparison with the transformer-based models, we also construct an MLP-based autoencoder to pre-train the stellar spectra. For both LAMOST LRS and Gaia XP, the network architectures are similar: an initial projection layer of shape [ input dim , hidden dim ] ; the encoder consists of two MLP blocks, each with structure [ hidden dim , 3 × hidden dim , hidden dim ] ; followed by a bottleneck layer of shape [ hidden dim , 768] . The decoder mirrors this structure to reconstruct the input spectra. For LAMOST LRS, we use input dim = 1462 + 1 and hidden dim = 1245 ; for Gaia XP, we use input dim = 343 + 2 and hidden dim = 1290 . The former includes one additional (logarithmic) standard deviation as the first token, while the latter includes both the mean and the standard deviation as the first two tokens. We train the LRS model on 4 GPUs for 128 epochs, taking approximately 140 minutes, and the Gaia XP model 4 GPUs for 191 epochs, taking approximately 150 minutes.

## E. PROJECTION MODELS AND DECODERS

This section provides details about the modules used in CLIP-based model training. For these models, we use 8 GPU and require a total of roughly 3 hours for training. All model variants use a batch size of 1024 per GPU for contrastive training with decoders. We adopt the AdamW optimizer with a learning rate of 1 × 10 -4 and a weight decay of 0.05. The learning rate is scheduled using a cosine annealing schedule with linear warm-up.

## E.1. Projection Networks

For the projection networks, the LAMOST LRS model follows Parker et al. (2024). We first obtain the output from a pre-trained LAMOST LRS model, then go through a cross-attention module with a learnable query vector. The cross-attention module has four attention heads with an output dimension of 768. Finally, we have an MLP layer with hidden features that have dimension 4 × 768 . We did not compress the information more in this projected network in order to investigate the information gain without compression. Therefore, the dimension of the final projected embedding is still 768. The number of trainable parameters is about 7.1 million.

For the Gaia XP spectra pre-trained with the transformer-based spectral model, we use the same projection networks as for the LAMOST LRS model. For the Gaia XP spectra pre-trained with a MLP-based autoencoder, our projection network has the dimension of [768 , 768 , 1160 , 768] , with a residual MLP block at the end with hidden dimension 4 × 768 . The choice of number of layers and the dimension of each layer is arbitrary; the key control condition is to maintain the same number of total trainable parameters as its cross-attention counterpart (7.1 million).

For the CLIP-split model, the LAMOST LRS projection network (5.1 million) outputs two branches for shared and non-shared representations:

- Shared : uses CrossAttentionHead with 512-D projection and n = 4 heads.
- Non-shared : uses CrossAttentionHead with 256-D projection and n = 2 heads.

Both branches use MLPs with hidden size 4 × 768 .

The Gaia XP projection network paired with the transformer-based pre-trained model has the same architecture as the LAMOST LRS projection network. For the projection network (also 5.1 million trainable parameters) paired with MLP-based pretrained model, it outputs two latent representations:

- Shared : linear projection to 512-D, followed by MLP: [512, 1160, 512].
- Non-shared : linear projection to 256-D, followed by MLP: [256, 1160, 256].

Each pathway includes a residual MLP block at the end with hidden dimension of 4 × 512 and 4 × 256 , respectively.

## E.2. Decoders

For the CLIP-pr model, the XP Decoder and LRS → XP cross decoder share the same architecture with MLP layer dimensions: [in, 4 × in, 2 × in, in, out] (8.5 million). The LRS decoder and XP → LRS cross decoder share the same architecture with layer dimensions: [in, 4 × in, 4 × in, 4 × in, out] (25.8 million), where in = 768 ; out = 1462 and 343 for LAMOST LRS and Gaia XP, respectively. Note that for LAMOST LRS, we are only reconstructing the normalized spectra with the mean and standard deviation calculated for each spectra, similar for the CLIP-split model.

For the CLIP-split model, both LRS (14.0 million) and XP (0.1 million) decoders take shared and non-shared features to reconstruct the original spectra. Shared and non-shared inputs are projected to the out dimension, where out = 1462 and 343 for LAMOST LRS and Gaia XP, respectively. The two outputs are concatenated, and the final reconstruction layer contains [out × 2, out × 2, out].

For the cross-modal decoder of the CLIP-split model, we take only the shared representation as input. The decoder architecture depends on the output dimension, yielding approximately 12.5 and 3.9 million trainable parameters for the following two setups:

- For LAMOST (out &gt; shared) : [512, 2048, 2048, 2048, out].
- For Gaia XP (out ≤ shared) : [512, 2048, 1024, 512, out].

## F. LOSS CURVES

This section presents the comparisons of the loss curves between the CLIP-pr and CLIP-split models, as shown in Figure F3. Although the CLIP-pr model achieves a lower CLIP loss during training, it exhibits a lower absolute cosine similarity in test pairs, as shown in Table 2. This discrepancy may arise from two factors. First, the geometry of the high-dimensional embedding spaces-768 dimensions in the CLIP-pr projected embedding versus 512 in the CLIP-split shared embedding-tends to yield lower cosine similarity in the former. Second, the CLIP loss focuses on relative alignment rather than absolute similarity. Additionally, the CLIP-pr model yields slightly lower cross-modal prediction losses. In contrast, the CLIP-split model converges more quickly in learning the reconstruction and achieves marginally lower reconstruction loss, likely due to its design of a non-shared projected embedding space.

Figure F3. Loss curves of the CLIP-pr and CLIP-split models.

## G. SUMMARY OF KEY HYPER-PARAMETERS AND CONFIGURATIONS

Table G2 summarizes the key hyper-parameters and configurations used across the main training stages. For complete details and implementation settings, please refer to Appendix D, Appendix E, and Section 2.

Table G2. Summary of hype-parameters and configurations across main training stages.

| CLIP temp   | Loss weights             | Epochs / Time Loss fn.                           | GPUs   | Batch / GPU       | Learning rate Weight decay LR schedule   | Masking / Rate Optimizer   | Latent dim. Tokenization              | Encoder / Layers                     | Setting              |            | Loss fn. Loss weights CLIP temp τ                                | Epochs / Time                 | LR schedule Batch / GPU                        | rate decay          | Learning Weight            | Latent dim. Tokenization Chunks Masking / Rate Optimizer   | Encoder / Layers                    | Setting     |
|-------------|--------------------------|--------------------------------------------------|--------|-------------------|------------------------------------------|----------------------------|---------------------------------------|--------------------------------------|----------------------|------------|------------------------------------------------------------------|-------------------------------|------------------------------------------------|---------------------|----------------------------|------------------------------------------------------------|-------------------------------------|-------------|
| 15.5        | w recon = w pred = 1 . 0 | 110 / 3 h L CLIP + w recon L recon + w pred L    | 1024 8 | Cosine + warm-up  | 1 × 10 - 4 0.05                          | - AdamW                    | 768 Same as CLIP                      | Cross-Attn (4 heads)+MLP             | CLIP-pr              |            | Masked MSE - -                                                   | 128 / 20 h                    | Cosine + warm-up 64                            | 1 × 10 - 5 0.10     | Chunk-based ( ∼ 45%) AdamW | 768 ( L =20 , O =10 )+ log 10 σ                            | 6 Self-Attn (6 heads) 2             | LRSMT       |
| 15.5        | w recon = w pred = 1 . 0 | 110 / 3 h pred L CLIP + w recon L recon + w pred | 1024 8 | Cosine + warm-up  | 1 × 10 - 4 0.05                          | - AdamW                    | 512+256 Same as CLIP                  | 2 Cross-Attn branches                | CLIP-split           | (Continue) | - - - 15.5                                                       | 191 / 2.5 h 110 /3 MSE L CLIP | Cosine + warm-up Cosine + 64 1024              | 1 × 10 - 5 1 × 0.10 | - AdamW AdamW              | 768 768 Prepend µ,σ Use pre-trained -                      | MLP blocks enc.+dec. Cross-Attn (4  | XP OAE CLIP |
| -           | -                        | Max. 100 / < 10 min L pred MSE                   | 32 1   | ReduceLROnPlateau | 1 × 1 × 10 - 4                           | - AdamW                    | Raw spectra /                         | 4 MLP [in,1024,512,64,1]             | Downstream MLP       |            | 15.5                                                             | h L CLIP + w recon            | warm-up Cosine + 1024                          | 10 - 4 1 0.05       |                            | 768 embeddings Same as                                     | heads)+MLP Cross-Attn (4            | CLIP-r      |
| -           | -                        | - / < 10 min Negative log-likelihood             | 50 1   | -                 | 10 - 5 5 × 10 -                          | - Adam - 4                 | - embeddings Raw spectra / embeddings | 2 transforms, 60 hidden units each - | Downstream SBI (NPE) |            | recon L CLIP + w pred L pred w recon = 1 . 0 w pred = 1 . 0 15.5 | 110 / 3 h 110 / 3 h L         | 1 × 10 0.05 0.05 warm-up Cosine + warm-up 1024 | × 10 - 4            | - AdamW AdamW - 4          | 768 CLIP Same as CLIP -                                    | heads)+MLP Cross-Attn (4 heads)+MLP | CLIP-p      |

## REFERENCES

Abazajian, K., Adelman-McCarthy, J. K., Ag¨ ueros, M. A., et al. 2003, AJ, 126, 2081, doi: 10.1086/378165

Euclid Collaboration, Siudek, M., Huertas-Company, M., et al. 2025, arXiv e-prints, arXiv:2503.15312,

Abdurro'uf, Accetta, K., Aerts, C., et al. 2022, ApJS, 259, 35, doi: 10.3847/1538-4365/ac4414

Andrae, R., Rix, H.-W., &amp; Chandra, V. 2023, ApJS, 267, 8, doi: 10.3847/1538-4365/acd53e

Astropy Collaboration, Robitaille, T. P., Tollerud, E. J., et al. 2013, A&amp;A, 558, A33, doi: 10.1051/0004-6361/201322068

Astropy Collaboration, Price-Whelan, A. M., Sip˝ ocz, B. M., et al. 2018, AJ, 156, 123, doi: 10.3847/1538-3881/aabc4f

Barber, D., &amp; Agakov, F. 2003, in Proceedings of the 17th International Conference on Neural Information Processing Systems, NIPS'03 (Cambridge, MA, USA: MIT Press), 201-208

Brown, T., Mann, B., Ryder, N., et al. 2020, in Advances in Neural Information Processing Systems, ed. H. Larochelle, M. Ranzato, R. Hadsell, M. Balcan, &amp; H. Lin, Vol. 33 (Curran Associates, Inc.), 1877-1901. https://proceedings.neurips.cc/paper files/ paper/2020/file/1457c0d6bfcb4967418bfb8ac142f64a-Paper.pdf

Buck, T., &amp; Schwarz, C. 2024, arXiv e-prints, arXiv:2410.16081, doi: 10.48550/arXiv.2410.16081

Buder, S., Kos, J., Wang, X. E., et al. 2025, PASA, 42, e051, doi: 10.1017/pasa.2025.26

Chaplin, W. J., Basu, S., Huber, D., et al. 2014, ApJS, 210, 1, doi: 10.1088/0067-0049/210/1/1

Cui, X.-Q., Zhao, Y.-H., Chu, Y.-Q., et al. 2012, Research in Astronomy and Astrophysics, 12, 1197, doi: 10.1088/1674-4527/12/9/003

Cunningham, H., Ewart, A., Riggs, L., Huben, R., &amp; Sharkey, L. 2023, arXiv e-prints, arXiv:2309.08600, doi: 10.48550/arXiv.2309.08600

De Angeli, F., Weiler, M., Montegriffo, P., et al. 2023, A&amp;A, 674, A2, doi: 10.1051/0004-6361/202243680

de Jong, J. T. A., Yanny, B., Rix, H.-W., et al. 2010, ApJ, 714, 663, doi: 10.1088/0004-637X/714/1/663

De Silva, G. M., Freeman, K. C., Bland-Hawthorn, J., et al. 2015, MNRAS, 449, 2604, doi: 10.1093/mnras/stv327

DESI Collaboration, Aghamousa, A., Aguilar, J., et al. 2016, arXiv e-prints, arXiv:1611.00036, doi: 10.48550/arXiv.1611.00036

DESI Collaboration, Abdul-Karim, M., Adame, A. G., et al. 2025, arXiv e-prints, arXiv:2503.14745, doi: 10.48550/arXiv.2503.14745

Devlin, J., Chang, M.-W., Lee, K., &amp; Toutanova, K. 2018, arXiv e-prints, arXiv:1810.04805, doi: 10.48550/arXiv.1810.04805

Devon Hjelm, R., Fedorov, A., Lavoie-Marchildon, S., et al. 2018, arXiv e-prints, arXiv:1808.06670, doi: 10.48550/arXiv.1808.06670

Durkan, C., Bekasov, A., Murray, I., &amp; Papamakarios, G. 2019, arXiv e-prints, arXiv:1906.04032, doi: 10.48550/arXiv.1906.04032

doi: 10.48550/arXiv.2503.15312

Fitzpatrick, M. J., Olsen, K., Economou, F., et al. 2014, in Observatory Operations: Strategies, Processes, and Systems V, ed. A. B. Peck, C. R. Benn, &amp; R. L. Seaman, Vol. 9149, International Society for Optics and Photonics (SPIE), 91491T, doi: 10.1117/12.2057445

Freeman, K., &amp; Bland-Hawthorn, J. 2002, ARA&amp;A, 40, 487, doi: 10.1146/annurev.astro.40.060401.093840

Gaia Collaboration, Vallenari, A., Brown, A. G. A., et al. 2023, A&amp;A, 674, A1, doi: 10.1051/0004-6361/202243940

Gilmore, G., Wyse, R. F. G., &amp; Kuijken, K. 1989, ARA&amp;A, 27, 555, doi: 10.1146/annurev.aa.27.090189.003011

Gneiting, T., Balabdaoui, F., &amp; Raftery, A. E. 2007, Journal of the Royal Statistical Society Series B: Statistical Methodology, 69, 243, doi: 10.1111/j.1467-9868.2007.00587.x

Gray, J., Szalay, A. S., Thakar, A. R., Stoughton, C., &amp; vandenBerg, J. 2002, in Virtual Observatories, ed. A. S. Szalay, Vol. 4846, International Society for Optics and Photonics (SPIE), 103 - 107, doi: 10.1117/12.461524

Hackstein, J., Sumbul, G., Clasen, K. N., &amp; Demir, B. 2024, arXiv e-prints, arXiv:2401.07782, doi: 10.48550/arXiv.2401.07782

Helmi, A. 2020, ARA&amp;A, 58, 205, doi: 10.1146/annurev-astro-032620-021917

Helou, G., Madore, B. F., Schmitz, M., et al. 1991, in Astrophysics and Space Science Library, Vol. 171, Databases and On-line Data in Astronomy, ed. M. A. Albrecht &amp; D. Egret, 89-106, doi: 10.1007/978-94-011-3250-3 10

Ho, M., Bartlett, D. J., Chartier, N., et al. 2024, The Open Journal of Astrophysics, 7, 54, doi: 10.33232/001c.120559

Hoaglin, D. C., Mosteller, F., &amp; Tukey, J. W. 1983, Understanding Robust and Exploratory Data Analysis (New York: John Wiley &amp;Sons)

Huang, Y., Beers, T. C., Xiao, K., et al. 2024, ApJ, 974, 192, doi: 10.3847/1538-4357/ad6b94

J¨ onsson, H., Holtzman, J. A., Allende Prieto, C., et al. 2020, AJ, 160, 120, doi: 10.3847/1538-3881/aba592

Jumper, J., Evans, R., Pritzel, A., et al. 2021, nature, 596, 583 Kaplan, J., McCandlish, S., Henighan, T., et al. 2020, CoRR, abs/2001.08361. https://arxiv.org/pdf/2001.08361.pdf

Koleva, M., Prugniel, P., Bouchard, A., &amp; Wu, Y. 2009, A&amp;A, 501, 1269, doi: 10.1051/0004-6361/200811467

Kolmogorov, A. 1992, On the Empirical Determination of a Distribution Function, ed. S. Kotz &amp; N. L. Johnson (New York,

NY: Springer New York), 106-113, doi: 10.1007/978-1-4612-4380-9 10

Koposov, S. E., Li, T. S., Allende Prieto, C., et al. 2025, arXiv e-prints, arXiv:2505.14787, doi: 10.48550/arXiv.2505.14787

- Lee, Y. S., Beers, T. C., Sivarani, T., et al. 2008, AJ, 136, 2022, doi: 10.1088/0004-6256/136/5/2022
- Lee, Y. S., Beers, T. C., Carlin, J. L., et al. 2015, AJ, 150, 187, doi: 10.1088/0004-6256/150/6/187
- Leung, H. W., &amp; Bovy, J. 2024, MNRAS, 527, 1494, doi: 10.1093/mnras/stad3015
- Li, C.-q., Shi, J.-r., Yan, H.-l., et al. 2024a, ApJS, 273, 18, doi: 10.3847/1538-4365/ad5002
- Li, H., Aoki, W., Matsuno, T., et al. 2022a, ApJ, 931, 147, doi: 10.3847/1538-4357/ac6514
- Li, J., Wong, K. W. K., Hogg, D. W., Rix, H.-W., &amp; Chandra, V. 2024b, ApJS, 272, 2, doi: 10.3847/1538-4365/ad2b4d
- Li, T., Li, Y ., Bi, S., et al. 2022b, ApJ, 927, 167, doi: 10.3847/1538-4357/ac4fbf
- Li, X., &amp; Lin, B. 2023, MNRAS, 521, 6354, doi: 10.1093/mnras/stad831
- Loshchilov, I., &amp; Hutter, F. 2017, arXiv e-prints, arXiv:1711.05101, doi: 10.48550/arXiv.1711.05101
- Luo, A. L., Zhao, Y.-H., Zhao, G., et al. 2015, Research in Astronomy and Astrophysics, 15, 1095, doi: 10.1088/1674-4527/15/8/002
- Majewski, S. R., Schiavon, R. P., Frinchaboy, P. M., et al. 2017, AJ, 154, 94, doi: 10.3847/1538-3881/aa784d
- Margon, B. 1999, Philosophical Transactions of the Royal Society of London Series A, 357, 93, doi: 10.1098/rsta.1999.0316
- Moitinho, A., Krone-Martins, A., Savietto, H., et al. 2017, A&amp;A, 605, A52, doi: 10.1051/0004-6361/201731059
- Moultaka, J., Ilovaisky, S. A., Prugniel, P., &amp; Soubiran, C. 2004, PASP, 116, 693, doi: 10.1086/422177
- Ness, M., Hogg, D. W., Rix, H. W., Ho, A. Y. Q., &amp; Zasowski, G. 2015, ApJ, 808, 16, doi: 10.1088/0004-637X/808/1/16
- OMullane, W., Li, N., Nieto-Santisteban, M., et al. 2005, arXiv e-prints, cs/0502072, doi: 10.48550/arXiv.cs/0502072
- Papamakarios, G., Pavlakou, T., &amp; Murray, I. 2017, arXiv e-prints, arXiv:1705.07057, doi: 10.48550/arXiv.1705.07057
- Parker, L., Lanusse, F., Golkar, S., et al. 2024, MNRAS, 531, 4990, doi: 10.1093/mnras/stae1450
- Radford, A., Wu, J., Child, R., et al. 2019, Language models are unsupervised multitask learners.

https://www.semanticscholar.org/paper/ Language-Models-are-Unsupervised-Multitask-Learners-Radford-Wu/ 9405cc0d6169988371b2755e573cc28650d14dfe

- Radford, A., Kim, J. W., Hallacy, C., et al. 2021, arXiv e-prints, arXiv:2103.00020, doi: 10.48550/arXiv.2103.00020
- Rix, H.-W., Chandra, V., Andrae, R., et al. 2022, ApJ, 941, 45, doi: 10.3847/1538-4357/ac9e01

Rizhko, M., &amp; Bloom, J. S. 2024, arXiv e-prints, arXiv:2411.08842, doi: 10.48550/arXiv.2411.08842

- R´ o˙ za´ nski, T., &amp; Ting, Y .-S. 2025, arXiv e-prints, arXiv:2503.18617, doi: 10.48550/arXiv.2503.18617
- R´ o˙ za´ nski, T., Ting, Y .-S., &amp; Jabło´ nska, M. 2025, ApJ, 980, 66, doi: 10.3847/1538-4357/ad9b99
- Sestito, F., Longeard, N., Martin, N. F., et al. 2019, MNRAS, 484, 2166, doi: 10.1093/mnras/stz043
- Shwartz Ziv, R., &amp; LeCun, Y. 2024, Entropy, 26, 252, doi: 10.3390/e26030252
- Smith, M. J., Roberts, R. J., Angeloudi, E., &amp; Huertas-Company, M. 2024, arXiv e-prints, arXiv:2405.14930, doi: 10.48550/arXiv.2405.14930
- Steinmetz, M., Zwitter, T., Siebert, A., et al. 2006, AJ, 132, 1645, doi: 10.1086/506564
- Sui, C., Zhao, X., Jing, T., &amp; Mao, Y. 2023, in Machine Learning for Astrophysics, 30, doi: 10.48550/arXiv.2307.04994
- Szalay, A., &amp; Gray, J. 2001, Science, 293, 2037, doi: 10.1126/science.293.5537.2037
- Szalay, A., Gray, J., Thakar, A., et al. 2001, arXiv e-prints, cs/0111015, doi: 10.48550/arXiv.cs/0111015
- Takada, M., Ellis, R. S., Chiba, M., et al. 2014, PASJ, 66, R1, doi: 10.1093/pasj/pst019
- Talts, S., Betancourt, M., Simpson, D., Vehtari, A., &amp; Gelman, A. 2018, arXiv preprint arXiv:1804.06788
- Tejero-Cantero, A., Boelts, J., Deistler, M., et al. 2020, Journal of Open Source Software, 5, 2505, doi: 10.21105/joss.02505

Ting, Y.-S. 2024, arXiv e-prints, arXiv:2412.05806, doi: 10.48550/arXiv.2412.05806

Paszke, A., Gross, S., Massa, F., et al. 2019, in Advances in Neural Information Processing Systems 32 (Curran Associates, Inc.), 8024-8035. http://papers.neurips.cc/paper/

- -. 2025, arXiv e-prints, arXiv:2506.12230, doi: 10.48550/arXiv.2506.12230
- Ting, Y.-S., Conroy, C., Rix, H.-W., &amp; Cargile, P. 2019, ApJ, 879, 69, doi: 10.3847/1538-4357/ab2331

9015-pytorch-an-imperative-style-high-performance-deep-learning-library. pdf

- Ting, Y.-S., Rix, H.-W., Conroy, C., Ho, A. Y. Q., &amp; Lin, J. 2017, ApJL, 849, L9, doi: 10.3847/2041-8213/aa921c
- Pattnaik, R., Kartaltepe, J. S., &amp; Binu, C. 2025, arXiv e-prints, arXiv:2501.01070, doi: 10.48550/arXiv.2501.01070

Poole, B., Ozair, S., van den Oord, A., Alemi, A. A., &amp; Tucker, G. 2019, arXiv e-prints, arXiv:1905.06922, doi: 10.48550/arXiv.1905.06922

- Radford, A., Narasimhan, K., Salimans, T., &amp; Sutskever, I. 2018

Vaswani, A., Shazeer, N., Parmar, N., et al. 2017, in Advances in Neural Information Processing Systems, ed. I. Guyon, U. V. Luxburg, S. Bengio, H. Wallach, R. Fergus, S. Vishwanathan, &amp; R. Garnett, Vol. 30 (Curran Associates, Inc.).

https://proceedings.neurips.cc/paper files/paper/2017/file/ 3f5ee243547dee91fbd053c1c4a845aa-Paper.pdf

- Viswanathan, A., Starkenburg, E., Matsuno, T., et al. 2024, A&amp;A, 683, L11, doi: 10.1051/0004-6361/202347944
- Vrard, M., Mosser, B., &amp; Samadi, R. 2016, A&amp;A, 588, A87, doi: 10.1051/0004-6361/201527259
- Wang, H., Guo, X., Deng, Z.-H., &amp; Lu, Y. 2022, arXiv e-prints, arXiv:2203.07004, doi: 10.48550/arXiv.2203.07004
- Wang, R., Luo, A. L., Zhang, S., et al. 2023, ApJS, 266, 40, doi: 10.3847/1538-4365/acce36
- Wu, Y., Du, B., Luo, A., Zhao, Y., &amp; Yuan, H. 2014, in IAU Symposium, Vol. 306, Statistical Challenges in 21st Century Cosmology, ed. A. Heavens, J.-L. Starck, &amp; A. Krone-Martins, 340-342, doi: 10.1017/S1743921314010825
- Wyse, R. F. G. 2009, in IAU Symposium, Vol. 258, The Ages of Stars, ed. E. E. Mamajek, D. R. Soderblom, &amp; R. F. G. Wyse, 11-22, doi: 10.1017/S1743921309031664
- Xiang, M., Ting, Y.-S., Rix, H.-W., et al. 2019, ApJS, 245, 34, doi: 10.3847/1538-4365/ab5364
- Xiang, M. S., Liu, X. W., Yuan, H. B., et al. 2015, MNRAS, 448, 822, doi: 10.1093/mnras/stu2692
- Yuan, H. B., Liu, X. W., &amp; Xiang, M. S. 2013, MNRAS, 430, 2188, doi: 10.1093/mnras/stt039
- Zhang, M., Xiang, M., Ting, Y.-S., et al. 2025, arXiv e-prints, arXiv:2506.02763, doi: 10.48550/arXiv.2506.02763
- Zhao, G., Zhao, Y.-H., Chu, Y.-Q., Jing, Y.-P., &amp; Deng, L.-C. 2012, Research in Astronomy and Astrophysics, 12, 723, doi: 10.1088/1674-4527/12/7/002
- Zhao, X., &amp; Huang, Y. 2025, SpecCLIP v1.0.0: Aligning and Translating Spectroscopic Measurements for Stars, v1.0.0, Zenodo, doi: 10.5281/zenodo.17824840
- Zhao, X., Li, X., Li, H., &amp; Liu, X. 2025, ApJS, 278, 41, doi: 10.3847/1538-4365/adcf9b
- Zhong, F., Napolitano, N. R., Heneka, C., et al. 2024, arXiv e-prints, arXiv:2412.21130, doi: 10.48550/arXiv.2412.21130

### UnstructuredLoader
- 다양한 비정형 문서들을 읽어 오는 Unstrctured 를 사용해, 다양한 형식의 문서들을 load 해 RAG, 모델 파인튜닝에 적용할 수있게 한다.
  - 지원 파일 형식: "csv", "doc", "docx", "epub", "image", "md", "msg", "odt", "org", "pdf", "ppt", "pptx", "rtf", "rst", "tsv", "xlsx"
- **다양한 형식의 파일로 부터 text를 로딩**해야 할 경우 유용하다. 
- Local에 library를 설치해서 사용하거나,  Unstructured 가 제공하는 API service를 사용할 수 있다.
  - https://docs.unstructured.io
- 텍스트 파일, PDF, 이미지, HTML, XML, ms-office(word, ppt), epub 등 다양한 비정형 데이터 파일을 처리할 수 있다.
  - 설치, 지원 문서: https://docs.unstructured.io/open-source/installation/full-installation
  - Langchain 문서: https://python.langchain.com/docs/integrations/document_loaders/unstructured_file

> - UnstructuredLoader PDF Load 시 Document 분할 기준
>     -  문서의 구조와 콘텐츠를 기반으로 텍스트를 분할해 Document에 넣는다.
>     -  분할 기준
>        - 헤더(Header): 문서의 제목이나 섹션 제목 등
>        - 본문 텍스트(NarrativeText): 일반적인 문단이나 설명문
>        - 표(Table): 데이터가 표 형식으로 구성된 부분
>        - 리스트(List): 순서가 있거나 없는 목록
>        - 이미지(Image): 사진이나 그래픽 요소

#### 설치할 프로그램
- poppler
  - pdf 파일을 text로 변환하기 위해 필요한 프로그램
  - windows: https://github.com/oschwartz10612/poppler-windows/releases/ 에서 최신 버전 다운로드 후 압축 풀어서 설치.
    - 환경변수 Path에 "설치경로\Library\bin" 을 추가. (설치 후 IDE를 다시 시작한다.)
  - macOS: `brew install poppler`
  - Linux: `sudo apt-get install poppler-utils`
- tesseract-ocr
  - OCR 라이브러리로 pdf 이미지를 text로 변환하기 위해 필요한 프로그램 
  - windows: https://github.com/UB-Mannheim/tesseract/wiki 에서 다운받아 설치. 
    - 환경변수 Path에 설치 경로("C:\Program Files\Tesseract-OCR") 추가 한다. (설치 후 IDE를 다시 시작한다.)
  - macOS: `brew install tesseract`
  - linux(unbuntu): `sudo apt install tesseract-ocr`
- 설치 할 패키지
  - **libmagic 설치**
      - windows: `pip install python-magic-bin -qU`
      - macOS: `brew install libmagic`
      - linux(ubuntu): `sudo apt-get install libmagic-dev`
  - `pip install "unstructured[pdf]" -qU`
      - 문서 형식별로 sub module을 설치한다. (pdf, docx ..)
      - 모든 sub module 설치: `pip install unstructured[all-docs]`
      - https://docs.unstructured.io/open-source/installation/full-installation
  - `pip install langchain-unstructured -qU`

In [ ]:
# !uv pip install python-magic-bin
# !uv pip install unstructured[all-docs]
# !uv pip install langchain-unstructured

Resolved 41 packages in 144ms
Prepared 2 packages in 462ms
Uninstalled 1 package in 32ms
Installed 2 packages in 427ms
 + langchain-unstructured==1.0.0
 - onnxruntime==1.23.2
 + onnxruntime==1.19.2


In [ ]:
from langchain_unstructured import UnstructuredLoader

path = ['data/olympic_wiki.md', "data/novel/메밀꽃_필_무렵_이효석.pdf"]

loader = UnstructuredLoader(path)
docs = loader.load() # 문단 단위로 문서를 split해서 Document에 넣어 제공.

INFO: pikepdf C++ to Python logger bridge initialized


In [6]:
len(docs)

244

In [7]:
docs[0].metadata

{'source': 'data/olympic_wiki.md',
 'category_depth': 0,
 'languages': ['kor'],
 'file_directory': 'data',
 'filename': 'olympic_wiki.md',
 'filetype': 'text/markdown',
 'last_modified': '2025-12-12T17:37:26',
 'category': 'Title',
 'element_id': '869efdd92ae840d110075ad507174066'}

In [10]:
print(docs[1].page_content)

올림픽(영어: Olympic Games, 프랑스어: Jeux olympiques)은 전 세계 각 대륙 각국에서 모인 수천 명의 선수가 참가해 여름과 겨울에 스포츠 경기를 하는 국제적인 대회이다. 전 세계에서 가장 큰 지구촌 최대의 스포츠 축제인 올림픽은 세계에서 가장 인지도있는 국제 행사이다. 올림픽은 2년마다 하계 올림픽과 동계 올림픽이 번갈아 열리며, 국제 올림픽 위원회(IOC)가 감독하고 있다. 또한 오늘날의 올림픽은 기원전 8세기부터 서기 5세기에 이르기까지 고대 그리스 올림피아에서 열렸던 올림피아 제전에서 비롯되었다. 그리고 19세기 말에 피에르 드 쿠베르탱 남작이 고대 올림피아 제전에서 영감을 얻어, 근대 올림픽을 부활시켰다. 이를 위해 쿠베르탱 남작은 1894년에 IOC를 창설했으며, 2년 뒤인 1896년에 그리스 아테네에서 제 1회 올림픽이 열렸다. 이때부터 IOC는 올림픽 운동의 감독 기구가 되었으며, 조직과 활동은 올림픽 헌장을 따른다. 오늘날 전 세계 대부분의 국가에서 올림픽 메달은 매우 큰 영예이며, 특히 올림픽 금메달리스트는 국가 영웅급의 대우를 받으며 스포츠 스타가 된다. 국가별로 올림픽 메달리스트들에게 지급하는 포상금도 크다. 대부분의 인기있는 종목들이나 일상에서 쉽게 접하고 즐길 수 있는 생활스포츠 종목들이 올림픽이라는 한 대회에서 동시에 열리고, 전 세계 대부분의 국가 출신의 선수들이 참여하는 만큼 전 세계 스포츠 팬들이 가장 많이 시청하는 이벤트이다. 2008 베이징 올림픽의 모든 종목 누적 시청자 수만 47억 명에 달하며, 이는 인류 역사상 가장 많은 수의 인구가 시청한 이벤트였다.


### Directory 내의 문서파일들 로딩
- DirectoryLoader 이용

In [11]:
# Unstructured 기반 - 관련 lib가 설치되어 있어야 한다.
from langchain_community.document_loaders import DirectoryLoader
loader = DirectoryLoader(
    path = "./data", # 문서파일들을 찾을 root directory
    glob=["*.pdf", "*.docx", "*.txt"], # 찾을 문서 파일 명의 패턴을 glob 패턴으로 지정. (생략하면 모든 파일)
    recursive=True, # False = path 경로에서만 찾는다. True : path의 하위경로도 모두 찾는다.
    show_progress=True, # 진행 프로그래스바가 나온다.
)

docs = loader.load()

  5%|▍         | 1/22 [00:01<00:21,  1.02s/it]WARNING: Could get FontBBox from font descriptor because None cannot be parsed as 4 floats


  9%|▉         | 2/22 [00:01<00:17,  1.14it/s]WARNING: Could get FontBBox from font descriptor because None cannot be parsed as 4 floats


 14%|█▎        | 3/22 [00:02<00:17,  1.11it/s]WARNING: Could get FontBBox from font descriptor because None cannot be parsed as 4 floats


 18%|█▊        | 4/22 [00:03<00:18,  1.01s/it]WARNING: Could get FontBBox from font descriptor because None cannot be parsed as 4 floats


 23%|██▎       | 5/22 [00:05<00:18,  1.07s/it]WARNING: Could get FontBBox from font descriptor because None cannot be parsed as 4 floats


...

 73%|███████▎  | 16/22 [01:05<00:20,  3.40s/it]

 77%|███████▋  | 17/22 [01:06<00:12,  2.56s/it]

 82%|████████▏ | 18/22 [01:07<00:08,  2.12s/it]

 95%|█████████▌| 21/22 [01:10<00:01,  1.27s/it]

100%|██████████| 22/22 [01:10<00:00,  3.21s/it]


In [13]:
print(len(docs))

22


In [15]:
docs[0].metadata

{'source': 'data\\novel\\금_따는_콩밭_김유정.pdf'}

In [14]:
docs[0].page_content

"1\n\n금 따는 콩밭\n\nExported from Wikisource on 2024년 11월 24일\n\n2\n\n위키백과에 이 글 과 관련된 자료가 있습니다. 금 따는 콩밭\n\n🙝🙟 땅속 저 밑은 늘 음침하 다.\n\n위키백과\n\n고달픈 간드렛불, 맥없이 푸르끼하다.\n\n밤과 달라서 낮엔 되우 흐릿하였다.\n\n겉으로 황토 장벽으로 앞뒤좌우가 콕 막힌 좁직한 구뎅이. 흡사히 무덤 속같이 귀중중하다. 싸늘한 침묵, 쿠더브레한 흙내와 징그러운 냉기만이 그 속에 자욱하다.\n\n곡괭이는 뻔질 흙을 이르집는다. 암팡스러이 내려쪼며,\n\n퍽 퍽 퍼억.\n\n이렇게 메떨어진 소리뿐. 그러나 간간 우수수 하고 벽이 헐 린다.\n\n영식이는 일손을 놓고 소맷자락을 끌어당기어 얼굴의 땀을 훑는다. 이놈의 줄이 언제나 잡힐는지 기가 찼다. 흙 한줌을 집어 코밑에 바짝 들여대고 손가락으로 샅샅이 뒤져본다. 완 연히 버력은 좀 변한 듯싶다. 그러나 불통버력이 아주 다 풀 린 것도 아니었다. 밀똥버력이라야 금이 온다는데 왜 이리 안 나오는지.\n\n곡괭이를 다시 집어든다. 땅에 무릎을 꿇고 궁뎅이를 번쩍 든 채 식식거린다. 곡괭이는 무작정 내려찍는다. 바닥에서\n\n3\n\n물이 스미어 무르팍이 흔건히 젖었다. 굿엎은 천판에서 흙방 울은 내리며 목덜미로 굴러든다. 어떤 때에는 웃벽의 한쪽이 떨어지며 등을 탕 때리고 부서진다.\n\n그러나 그는 눈도 하나 깜짝하지 않는다. 금을 캔다고 콩밭 하나를 다 잡쳤다. 약이 올라서 죽을둥 살둥 눈이 뒤집힌 이 판이다. 손바닥에 침을 탁 뱉고 곡괭이 자루를 한번 꼰아잡 더니 쉴 줄 모른다.\n\n등뒤에서는 흙 긁는 소리가 드윽드윽 난다. 아직도 버력을 다 못 친 모양. 이 자식이 일을 하나 시졸 하나. 남은 속이 바 직바직 타는데 웬 뱃심이 이리도 좋아.\n\n영식이는 살기 띤 시선으로 고개를 돌렸다. 암 말 없이 수재 를 노려본다. 그제야 꾸물꾸물 바지게에 흙을 담고 등에 메 고 사다리를 올라간다.\n\n굿이 풀리는지 벽이 우찔하

# Chunking (문서 분할)

![rag_split](figures/rag_split.png)

- Load 한 문서를 지정한 기준의 덩어리(chunk)로 나누는 작업을 진행한다.

## 나누는 이유
1. **임베딩 모델의 컨텍스트 길이 제한**
    - 대부분의 언어 모델은 한 번에 처리할 수 있는 토큰 수에 제한이 있다. 전체 문서를 통째로 입력하면 이 제한을 초과할 수 있어 처리가 불가능해진다.
2. **검색 정확도 향상**
    - 큰 문서 전체보다는 특정 주제나 내용을 다루는 작은 chunk가 사용자 질문과 더 정확하게 매칭된다. 예를 들어, 100페이지 매뉴얼에서 특정 기능에 대한 질문이 있을 때, 해당 기능을 설명하는 몇 개의 문단만 검색되는 것이 더 효과적이다.
    - 사용자 질문에 대해 문서의 모든 내용이 다 관련있는 것은 아니다. Chunking을 통해 가장 관련성 높은 부분만 선별적으로 활용할 수 있어 답변의 품질이 향상된다.
    - 전체 문서에는 질문과 무관한 내용들이 많이 포함되어 있어 모델이 혼란을 겪을 수 있다. 적절한 크기의 chunk는 이런 노이즈를 줄여준다.
3. **계산 효율성**
    - 벡터 유사도 계산, 임베딩 생성 등의 작업이 작은 chunk 단위로 수행될 때 더 빠르고 효율적이다. 메모리 사용량도 줄일 수 있다.

## 주요 Splitter
- **Splitter**는 문서를 분할(chunking)을 처리해주는 도구들이다. Langchain은 분할 대상, 방법에 따라 다양한 splitter를 제공한다.
- **Splitter 의 목표**
  - 가능한 한 **의미 있는 덩어리를 유지**하면서, **최대 길이(chunk_size)**를 넘지 않도록 나누기.
- https://reference.langchain.com/python/langchain_text_splitters/

### CharacterTextSplitter
가장  기본적인 Text spliter
- 한개의 구분자를 기준으로 분리한다. (default: "\n\n")
    - 분리된 조각이 chunk size 보다 작으면 다음 조각과 합칠 수 있다.
        - 합쳤을때 chuck_size 보다 크면 안 합친다. chuck_size 이내면 합친다.
    - 나누는 기준은 구분자이기 때문에 chunk_size 보다 글자수가 많을 수 있다.
- chunk size: 분리된 문서(chunk) 글자수 이내에서 분리되도록 한다.
    -  구분자를 기준으로 분리한다. 구분자를 기준으로 분리한 문서 조각이 chunk size 보다 크더라도 그대로 유지한다. 즉 chunk_size가 우선이 아니라 **seperator** 가 우선이다.
- 주요 파라미터
    - chunk_size: 각 조각의 최대 길이를 지정.
    - seperator: 구분 문자열을 지정. (default: '\n\n')
- CharacterTextSplitter는 단순 스플리터로 overlap기능을 지원하지는 않는다. 단 seperator가 빈문자열("") 일 경우에는 overlap 기능을 지원한다. overlap이란 각 이전 청크의 뒷부분의 문자열을 앞에 붙여 문맥을 유지하는 것을 말한다.
  
### RecursiveCharacterTextSplitter
- RecursiveCharacterTextSplitter는 **긴 텍스트를 지정된 최대 길이(chunk_size) 이하로 나누는 데 효과적인 텍스트 분할기**(splitter)이다.
- 여러 **구분자(separators)를 순차적으로 적용**하여, 가능한 한 자연스러운 문단/문장/단어 단위로 분할하고, 최종적으로는 크기 제한을 만족시킨다.
- 분할 기준 문자
    1. 두 개의 줄바꿈 문자 ("\n\n")
    2. 한 개의 줄바꿈 문자 ("\n")
    3. 공백 문자 (" ")
    4. 빈 문자열 ("")
- 작동 방식
    1. 먼저 가장 높은 우선순위의 구분자("\n\n")를 기준으로 분리한다.
    2. 분할된 조각 중 **chunk_size를 초과하는 조각**에 대해 다음 우선순위 구분자("\n" → " " → "")로 재귀적으로 재분할한다.
    3. 이 과정을 통해 모든 조각(chunk)이 chunk_size를 초과하지 않도록 만든다.  
- 주요 파라미터
    - chunk_size: 각 조각의 최대 길이를 지정.
    - chunk_overlap: 연속된 청크들 간의 겹치는 문자 수를 설정. 새로운 청크 생성 시 이전 청크의 마지막 부분에서 지정된 수만큼의 문자를 가져와서 새 청크의 앞부분에 포함시켜, 청크 경계에서 문맥의 연속성을 유지한다.
      - 구분자에 의해 청크가 나눠지면 정상적인 분리이므로 overlap이 적용되지 않는다.
      - 정상적 구분자로 나눌 수 없어 chunk_size에 맞춰 잘라진 경우 문맥의 연결성을 위애 overlap을 적용한다.
    - separators(list): 구분자를 지정한다. 지정하면 기본 구분자가 지정한 것으로 변경된다.

#### 메소드
- `split_documents(Iterable[Document]) : List[Document]`
    - Document 목록을 받아 split 처리한다.
- `split_text(str) : List[str]`
    - string text를 받아서 split 처리한다. 

In [ ]:
# !uv pip install langchain-text-splitters

Audited 1 package in 10ms


In [ ]:
from langchain_text_splitters import CharacterTextSplitter
from langchain_core.documents import Document

text = """123456789012345678901234567890123456789012345678901234567890123456789

abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ

가나다라마바사아자차카타파하

아야어여오요우유으이

abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ
"""


splitter = CharacterTextSplitter(
    chunk_size = 60,
    chunk_overlap=10, # default가 200, chunk size보다 overlap이 크면 안됨.
    # chunk_overlap은 60글자씩 잘랐을 때 중요한 내용이나 앞의 문맥을 파악할 수 있도록 앞에 글자를 붙여주는 것임(여기선 10글자)
    # → 이전 청크의 끝부분이 다음 청크의 시작 기준으로 재사용된다
    # separator="" # chunk size기준으로 나누기 > chunk overlap 적용
)

result = splitter.split_text(text)

print(len(result))

4


In [33]:
for txt in result :
    print(len(txt), txt, sep="-")

    # 가나다라는 왜 합쳐져있냐면 위에서 60으로 사이즈를 지정함.
    # > 그래서 위 문장과 합치면 60자가 넘기때문에 60자 이내면 다 합치려고 함.
    # > 맨 마지막 문장자체가 52자로 가나다라와 합치면 60을 훌쩍넘기때문에 가나다라-유으이 로만 묶인 것.

60-123456789012345678901234567890123456789012345678901234567890
60-1234567890123456789

abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLM
60-DEFGHIJKLMNOPQRSTUVWXYZ

가나다라마바사아자차카타파하

아야어여오요우유으이

abcdefg
55-이

abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ


In [22]:
# str -> Document 객체로 변환
doc = Document(page_content=text, metadata={"category":'split'})

docs2 = splitter.split_documents([doc])
len(docs2)

4

In [23]:
docs2[0]

Document(metadata={'category': 'split'}, page_content='123456789012345678901234567890123456789012345678901234567890123456789')

In [28]:
for doc in docs2 :
    print(len(doc.page_content))

69
52
26
52


In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text2 = """1234567890123456789012345678901234567890
12345678901234567890123456789

abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ

가나다라마바사아자차카타파하

아야어여오요우유으이

abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQ RSTUVWXYZ
abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ
"""

splitter = RecursiveCharacterTextSplitter(
    chunk_size=50,
    chunk_overlap=10
    seperators=["\n\n", '\n', '.', ' ', '']
    is_separator_regex=True # 위 seperator를 
)

result2 = splitter.split_text(text2)

for txt in result2 :
    print(len(txt), txt, sep="-")

40-1234567890123456789012345678901234567890
29-12345678901234567890123456789
49-abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVW
13-NOPQRSTUVWXYZ
26-가나다라마바사아자차카타파하

아야어여오요우유으이
43-abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQ
9-RSTUVWXYZ
49-abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVW
13-NOPQRSTUVWXYZ


In [ ]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
path = "data/olympic.txt"

# loading -> split
loader = TextLoader(path, encoding='utf-8')
splitter = RecursiveCharacterTextSplitter(
    chunk_size=500, chunk_overlap=20
)

load_docs = loader.load()
docs = splitter.split_documents(load_docs)
docs = [doc for doc in docs if len(doc.page_content) > 10]
len(docs)

61

In [38]:
print(docs[2].page_content)

따른다. 오늘날 전 세계 대부분의 국가에서 올림픽 메달은 매우 큰 영예이며, 특히 올림픽 금메달리스트는 국가 영웅급의 대우를 받으며 스포츠 스타가 된다. 국가별로 올림픽 메달리스트들에게 지급하는 포상금도 크다. 대부분의 인기있는 종목들이나 일상에서 쉽게 접하고 즐길 수 있는 생활스포츠 종목들이 올림픽이라는 한 대회에서 동시에 열리고, 전 세계 대부분의 국가 출신의 선수들이 참여하는 만큼 전 세계 스포츠 팬들이 가장 많이 시청하는 이벤트이다. 2008 베이징 올림픽의 모든 종목 누적 시청자 수만 47억 명에 달하며, 이는 인류 역사상 가장 많은 수의 인구가 시청한 이벤트였다.


In [40]:
docs = loader.load_and_split(splitter)
len(docs)

61

## Token 수 기준으로 나누기

- LLM 언어 모델들은 입력 토큰 수 제한이 있어서 요청시 제한 토큰수 이상의 프롬프트는 전송할 수 없다.
- 따라서 텍스트를 chunk로 분할할 때는 글자수 보다 **토큰 수를 기준으로 크기를 지정하는 것**이 좋다.  
- 토큰기반 분할은 텍스트의 의미를 유지하면서 분할하는 방식이므로 문자 기반 분할과 같이 단어가 중간잘리는 것들을 방지할 수 있다. 
- 토큰 수 계산할 때는 사용하는 언어 모델에 사용된 것과 동일한 tokenizer를 사용하는 것이 좋다.
  - 예를 들어 OpenAI의 GPT 모델을 사용할 경우 tiktoken 라이브러리를 활용하여 토큰 수를 정확하게 계산할 수 있다.

### [tiktoken](https://github.com/openai/tiktoken) tokenizer 기반 분할
- OpenAI에서 GPT 모델을 학습할 때 사용한 `BPE` 방식의 tokenizer. **OpenAI 언어모델을 사용할 경우 이것을 사용하는 것이 좀 더 정확하게  토큰을 계산할 수 있다.**
- Splitter.from_tiktoken_encoder() 메소드를 이용해 생성
  - `RecursiveCharacterTextSplitter.from_tiktoken_encoder()`
  - `CharacterTextSplitter.from_tiktoken_encoder()`
- 파라미터
  - encode_name: 인코딩 방식(토큰화 규칙)을 지정. OpenAI는 GPT 모델들 마다 다른 방식을 사용했다. 그래서 사용하려는 모델에 맞는 인코딩 방식을 지정해야 한다.
    - `cl100k_base`: GPT-4 및 GPT-3.5-Turbo 모델에서 사용된 방식.
    - `r50k_base:` GPT-3 모델에서 사용된 방식 
  - chunk_size, chunk_overlap, separators 파라미터 (위와 동일)
- tiktoken 설치
  - `pip install tiktoken`

### HuggingFace Tokenizer
- HuggingFace 모델을 사용할 경우 그 모델이 사용한 tokenizer를 이용해 토큰 기반으로 분할 한다.
  - 다른 tokenizer를 이용해 분할 할 경우 토큰 수 계산이 다르게 될 수있다.
- `from_huggingface_tokenizer()` 메소드를 이용.
  - 파라미터
    - tokenizer: HuggingFace tokenizer 객체
    - chunk_size, chunk_overlap, separators 파라미터 (위와 동일)
- `transformers` 라이브러리를 설치해야 한다.
  - `pip install transformers` 

In [2]:
# !uv pip show tiktoken
# !uv pip show transformers

Name: transformers
Version: 4.57.3
Location: C:\Users\Playdata\Documents\SKN21\JYS\10_langchain\.venv\Lib\site-packages
Requires: filelock, huggingface-hub, numpy, packaging, pyyaml, regex, requests, safetensors, tokenizers, tqdm
Required-by: docling-ibm-models, unstructured-inference


In [3]:
from langchain_text_splitters import RecursiveCharacterTextSplitter, CharacterTextSplitter
from langchain_community.document_loaders import TextLoader

path = "data/olympic.txt"
loader = TextLoader(path, encoding="utf-8")
splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    model_name="gpt-5-mini", # gpt-5-mini에서 사용한 tokenizer를 기준으로 한다.
    chunk_size = 500,
    chunk_overlap = 30
)

docs = loader.load_and_split(splitter)
len(docs)

36

In [4]:
print(docs[0].page_content)

올림픽
올림픽(영어: Olympic Games, 프랑스어: Jeux olympiques)은 전 세계 각 대륙 각국에서 모인 수천 명의 선수가 참가해 여름과 겨울에 스포츠 경기를 하는 국제적인 대회이다. 전 세계에서 가장 큰 지구촌 최대의 스포츠 축제인 올림픽은 세계에서 가장 인지도있는 국제 행사이다. 올림픽은 2년마다 하계 올림픽과 동계 올림픽이 번갈아 열리며, 국제 올림픽 위원회(IOC)가 감독하고 있다. 또한 오늘날의 올림픽은 기원전 8세기부터 서기 5세기에 이르기까지 고대 그리스 올림피아에서 열렸던 올림피아 제전에서 비롯되었다. 그리고 19세기 말에 피에르 드 쿠베르탱 남작이 고대 올림피아 제전에서 영감을 얻어, 근대 올림픽을 부활시켰다. 이를 위해 쿠베르탱 남작은 1894년에 IOC를 창설했으며, 2년 뒤인 1896년에 그리스 아테네에서 제 1회 올림픽이 열렸다. 이때부터 IOC는 올림픽 운동의 감독 기구가 되었으며, 조직과 활동은 올림픽 헌장을 따른다. 오늘날 전 세계 대부분의 국가에서 올림픽 메달은 매우 큰 영예이며, 특히 올림픽 금메달리스트는 국가 영웅급의 대우를 받으며 스포츠 스타가 된다. 국가별로 올림픽 메달리스트들에게 지급하는 포상금도 크다. 대부분의 인기있는 종목들이나 일상에서 쉽게 접하고 즐길 수 있는 생활스포츠 종목들이 올림픽이라는 한 대회에서 동시에 열리고, 전 세계 대부분의 국가 출신의 선수들이 참여하는 만큼 전 세계 스포츠 팬들이 가장 많이 시청하는 이벤트이다. 2008 베이징 올림픽의 모든 종목 누적 시청자 수만 47억 명에 달하며, 이는 인류 역사상 가장 많은 수의 인구가 시청한 이벤트였다.


In [5]:
# huggingface tokeinzer 사용
from transformers import AutoTokenizer
model_id = "google/gemma-3-4b-it"
tokenizer = AutoTokenizer.from_pretrained(model_id)
print(type(tokenizer))

tokenizer_config.json:   0%|          | 0.00/1.16M [00:00<?, ?B/s]

c:\Users\Playdata\Documents\SKN21\JYS\10_langchain\.venv\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Playdata\.cache\huggingface\hub\models--google--gemma-3-4b-it. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling

tokenizer.model:   0%|          | 0.00/4.69M [00:00<?, ?B/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


tokenizer.json:   0%|          | 0.00/33.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

<class 'transformers.models.gemma.tokenization_gemma_fast.GemmaTokenizerFast'>


In [6]:
splitter2 = RecursiveCharacterTextSplitter.from_huggingface_tokenizer(
    tokenizer,
    chunk_size = 500,
    chunk_overlap=20
)

docs = loader.load_and_split(splitter2)
len(docs)

35

In [7]:
print(docs[0].page_content)

올림픽
올림픽(영어: Olympic Games, 프랑스어: Jeux olympiques)은 전 세계 각 대륙 각국에서 모인 수천 명의 선수가 참가해 여름과 겨울에 스포츠 경기를 하는 국제적인 대회이다. 전 세계에서 가장 큰 지구촌 최대의 스포츠 축제인 올림픽은 세계에서 가장 인지도있는 국제 행사이다. 올림픽은 2년마다 하계 올림픽과 동계 올림픽이 번갈아 열리며, 국제 올림픽 위원회(IOC)가 감독하고 있다. 또한 오늘날의 올림픽은 기원전 8세기부터 서기 5세기에 이르기까지 고대 그리스 올림피아에서 열렸던 올림피아 제전에서 비롯되었다. 그리고 19세기 말에 피에르 드 쿠베르탱 남작이 고대 올림피아 제전에서 영감을 얻어, 근대 올림픽을 부활시켰다. 이를 위해 쿠베르탱 남작은 1894년에 IOC를 창설했으며, 2년 뒤인 1896년에 그리스 아테네에서 제 1회 올림픽이 열렸다. 이때부터 IOC는 올림픽 운동의 감독 기구가 되었으며, 조직과 활동은 올림픽 헌장을 따른다. 오늘날 전 세계 대부분의 국가에서 올림픽 메달은 매우 큰 영예이며, 특히 올림픽 금메달리스트는 국가 영웅급의 대우를 받으며 스포츠 스타가 된다. 국가별로 올림픽 메달리스트들에게 지급하는 포상금도 크다. 대부분의 인기있는 종목들이나 일상에서 쉽게 접하고 즐길 수 있는 생활스포츠 종목들이 올림픽이라는 한 대회에서 동시에 열리고, 전 세계 대부분의 국가 출신의 선수들이 참여하는 만큼 전 세계 스포츠 팬들이 가장 많이 시청하는 이벤트이다. 2008 베이징 올림픽의 모든 종목 누적 시청자 수만 47억 명에 달하며, 이는 인류 역사상 가장 많은 수의 인구가 시청한 이벤트였다.


## MarkdownHeaderTextSplitter
- Markdown Header 기준으로 Splitter
- Loading한 문서가 Markdown 문서이고 Header를 기준으로 문서의 내용이 나눠질때 사용.
- https://reference.langchain.com/python/langchain_text_splitters/#langchain_text_splitters.MarkdownTextSplitter

In [8]:
text = """
# 대주제1
- 동물

## 중주제1
- 포유류

- 조류

### 소주제1
- 개
- 고양이
- 까치
- 독수리

# 대주제2
## 중주제2
- 기차
- 배
"""

In [14]:
from langchain_text_splitters import MarkdownHeaderTextSplitter
# MarkdownHeaderTextSplitter는 split document method가 없음.
# header 정보는 metadata에 저장, 내용은 page content에 저장.
headers_to_split = [
    ("#", "Header1"),
    ("##", "Header2"),
    ("###", "Header3")
    # ("#", "대주제"),
    # ("##", "중주제"),
    # ("###", "소주제")
]
splitter = MarkdownHeaderTextSplitter(
    headers_to_split_on=headers_to_split, # 어떤 header를 기준으로 나눌지.
    strip_headers=True, # dafault가 True, header(#)을 내용에 포함시킬지 여부, False : 포함 안시킴
    return_each_line=False, # True : 각 라인을 별도의 document로 생성. , default False 
)

docs = splitter.split_text(text)
print(len(docs))

4


In [15]:
for doc in docs :
    print(doc.metadata)
    print(doc.page_content)
    print("=======")

{'Header1': '대주제1'}
- 동물
{'Header1': '대주제1', 'Header2': '중주제1'}
- 포유류  
- 조류
{'Header1': '대주제1', 'Header2': '중주제1', 'Header3': '소주제1'}
- 개
- 고양이
- 까치
- 독수리
{'Header1': '대주제2', 'Header2': '중주제2'}
- 기차
- 배


In [22]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import MarkdownHeaderTextSplitter

path = "data/olympic_wiki.md"

# Textloader대신 사용할 수 있는 방법.
# with open(path, "rt", encoding="utf-8") as fr:
#     content = fr.read()
# content
loader = TextLoader(path, encoding="utf-8")


headers_to_split_on = [
    ("##", "Header1"),
    ("###", "Header2"),
    ("####", "Header3")
]

splitter = MarkdownHeaderTextSplitter(
    headers_to_split_on=headers_to_split_on
)

In [23]:
load -> split
docs = loader.load() # [doc, doc, doc, ...]
doc_txt = '\n'.join(doc.page_content for doc in docs)
print(doc_txt)

# 올림픽

올림픽(영어: Olympic Games, 프랑스어: Jeux olympiques)은 전 세계 각 대륙 각국에서 모인 수천 명의 선수가 참가해 여름과 겨울에 스포츠 경기를 하는 국제적인 대회이다. 전 세계에서 가장 큰 지구촌 최대의 스포츠 축제인 올림픽은 세계에서 가장 인지도있는 국제 행사이다. 올림픽은 2년마다 하계 올림픽과 동계 올림픽이 번갈아 열리며, 국제 올림픽 위원회(IOC)가 감독하고 있다. 또한 오늘날의 올림픽은 기원전 8세기부터 서기 5세기에 이르기까지 고대 그리스 올림피아에서 열렸던 올림피아 제전에서 비롯되었다. 그리고 19세기 말에 피에르 드 쿠베르탱 남작이 고대 올림피아 제전에서 영감을 얻어, 근대 올림픽을 부활시켰다. 이를 위해 쿠베르탱 남작은 1894년에 IOC를 창설했으며, 2년 뒤인 1896년에 그리스 아테네에서 제 1회 올림픽이 열렸다. 이때부터 IOC는 올림픽 운동의 감독 기구가 되었으며, 조직과 활동은 올림픽 헌장을 따른다. 오늘날 전 세계 대부분의 국가에서 올림픽 메달은 매우 큰 영예이며, 특히 올림픽 금메달리스트는 국가 영웅급의 대우를 받으며 스포츠 스타가 된다. 국가별로 올림픽 메달리스트들에게 지급하는 포상금도 크다. 대부분의 인기있는 종목들이나 일상에서 쉽게 접하고 즐길 수 있는 생활스포츠 종목들이 올림픽이라는 한 대회에서 동시에 열리고, 전 세계 대부분의 국가 출신의 선수들이 참여하는 만큼 전 세계 스포츠 팬들이 가장 많이 시청하는 이벤트이다. 2008 베이징 올림픽의 모든 종목 누적 시청자 수만 47억 명에 달하며, 이는 인류 역사상 가장 많은 수의 인구가 시청한 이벤트였다.

또한 20세기에 올림픽 운동이 발전함에 따라, IOC는 변화하는 세계의 사회 환경에 적응해야 했다. 이러한 변화의 예로는 얼음과 눈을 이용한 경기 종목을 다루는 동계 올림픽, 장애인이 참여하는 패럴림픽, 스페셜 올림픽, 데플림픽, 10대 선수들이 참여하는 유스 올림픽 등을 들 수 있다. 그 뿐만 아니라 IOC는 20세기의 변화하는 경

In [24]:
split_docs = splitter.split_text(doc_txt)
len(split_docs)

29

In [26]:
split_docs[0]

Document(metadata={}, page_content='# 올림픽  \n올림픽(영어: Olympic Games, 프랑스어: Jeux olympiques)은 전 세계 각 대륙 각국에서 모인 수천 명의 선수가 참가해 여름과 겨울에 스포츠 경기를 하는 국제적인 대회이다. 전 세계에서 가장 큰 지구촌 최대의 스포츠 축제인 올림픽은 세계에서 가장 인지도있는 국제 행사이다. 올림픽은 2년마다 하계 올림픽과 동계 올림픽이 번갈아 열리며, 국제 올림픽 위원회(IOC)가 감독하고 있다. 또한 오늘날의 올림픽은 기원전 8세기부터 서기 5세기에 이르기까지 고대 그리스 올림피아에서 열렸던 올림피아 제전에서 비롯되었다. 그리고 19세기 말에 피에르 드 쿠베르탱 남작이 고대 올림피아 제전에서 영감을 얻어, 근대 올림픽을 부활시켰다. 이를 위해 쿠베르탱 남작은 1894년에 IOC를 창설했으며, 2년 뒤인 1896년에 그리스 아테네에서 제 1회 올림픽이 열렸다. 이때부터 IOC는 올림픽 운동의 감독 기구가 되었으며, 조직과 활동은 올림픽 헌장을 따른다. 오늘날 전 세계 대부분의 국가에서 올림픽 메달은 매우 큰 영예이며, 특히 올림픽 금메달리스트는 국가 영웅급의 대우를 받으며 스포츠 스타가 된다. 국가별로 올림픽 메달리스트들에게 지급하는 포상금도 크다. 대부분의 인기있는 종목들이나 일상에서 쉽게 접하고 즐길 수 있는 생활스포츠 종목들이 올림픽이라는 한 대회에서 동시에 열리고, 전 세계 대부분의 국가 출신의 선수들이 참여하는 만큼 전 세계 스포츠 팬들이 가장 많이 시청하는 이벤트이다. 2008 베이징 올림픽의 모든 종목 누적 시청자 수만 47억 명에 달하며, 이는 인류 역사상 가장 많은 수의 인구가 시청한 이벤트였다.  \n또한 20세기에 올림픽 운동이 발전함에 따라, IOC는 변화하는 세계의 사회 환경에 적응해야 했다. 이러한 변화의 예로는 얼음과 눈을 이용한 경기 종목을 다루는 동계 올림픽, 장애인이 참여하는 패럴림픽, 스페셜 올림픽, 데플림픽, 10대 선수들이 참여하는 유스 올